# speech2face (new pipeline) -- IMTalker Emotion LoRA Training

Adds **emotion conditioning** to the new speech2face pipeline's pretrained IMTalker
generator using targeted LoRA fine-tuning -- the same methodology proven in the old
IMTalker-PersonaPlex pipeline (`05_train_emotion_lora.ipynb`), adapted to the new
repository layout, models and dependencies.

After training you can make the avatar express **neutral, happy, sad, angry, surprise
or fear** by setting a single integer (`emotion_id`) at inference time.

## How this was adapted from the old pipeline

The old and new pipelines share the *same* IMTalker generator architecture
(`FMGenerator` + `FlowMatchingTransformer`). The new `src/imtalker/generator/` code is
byte-identical to the old `IMTalker/generator/` code **except that all emotion support
was removed** (verified by diffing the two repositories: -126 lines in `FM.py`,
-42 lines in `FMT.py`, -73 lines in `train_lora.py` -- all emotion code).

| Component | Old pipeline | New pipeline (this notebook) |
|---|---|---|
| Generator | `IMTalker/generator/FM.py` (with `EmotionConditioner`) | `src/imtalker/generator/FM.py` -> patched with the same `EmotionConditioner` |
| Transformer | `IMTalker/generator/FMT.py` (emotion bias in conditioning sum) | `src/imtalker/generator/FMT.py` -> patched identically |
| LoRA setup | `generator/train_lora.py` (`apply_emotion_lora_to_model`) | `src/imtalker/generator/train_lora.py` -> function appended |
| Dataset | `generator/emotion_dataset.py` | re-created identically for the new layout |
| Training | `generator/run_emotion_lora_training.py` | re-created + `--precision` (mixed precision support) |
| Per-epoch tests | `generator/imtalker_train_utils.py` | re-created identically |
| Final tests | `generator/test_emotions.py` | re-created identically |
| Base weights | `cbsjtu01/IMTalker` (generator.ckpt / renderer.ckpt / wav2vec2-base-960h) | same checkpoints |

The audio contract is also unchanged: the generator consumes **768-d wav2vec2-space
features**. In the new pipeline's live system those features are produced from PersonaPlex
**Helium** hidden states by the UniTalk last-layer adapter; for training, the emotion
dataset provides the same features precomputed in `audio/*.npy` (see the "Helium
feature extraction" section below for why no Helium pass is needed for training).

## What this notebook does

1. clones the emotion-adapter repository (`speech2face_orin_n_original_emotion_adapter_v2`)
2. installs all dependencies and detects the GPU (VRAM-based batch size, TF32, mixed precision)
3. writes the emotion adapter code into the cloned repo (patches `FM.py` / `FMT.py` /
   `train_lora.py`, adds `emotion_dataset.py`, `imtalker_train_utils.py`,
   `run_emotion_lora_training.py`, `test_emotions.py`)
4. downloads the pretrained IMTalker checkpoints (generator / renderer / wav2vec2)
5. verifies the emotion dataset (motion / audio / smirk / gaze + emotion labels)
6. launches emotion LoRA training (Option A: background process, multi-GPU DDP;
   Option B: in-notebook, single GPU)
7. renders a test video with `3robert.jpeg` + `audio_3.wav` **every 10 epochs**
   using the latest (EMA) weights -- without interrupting training
8. validates the trained checkpoint and renders all 6 emotions at the end

## Requirements

| | |
|---|---|
| **GPU** | NVIDIA GPU with CUDA (Ampere or newer recommended); multi-GPU DDP supported |
| **Disk** | ~10 GB free on `/workspace` (code + checkpoints + outputs) |
| **Dataset** | your emotion dataset in `/workspace/emotion_dataset` (flat layout `motion/ audio/ smirk/ gaze/`, same format as the old pipeline) |
| **Test files** | `3robert.jpeg` and `audio_3.wav` -- place them next to this notebook (auto-downloaded if missing) |


In [ ]:
# ============================================================================
# STEP 1 -- Clone the repository with the updated code
# ============================================================================
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL      = "https://github.com/MoshiHead/speech2face_orin_n_original_emotion_adapter_v2.git"
BASE_REPO_URL = "https://github.com/MoshiHead/speech2face_orin_n_original.git"   # base new-pipeline code
REPO_DIR      = "/workspace/speech2face_emotion"


def sh(cmd, cwd=None, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if r.stdout and r.stdout.strip():
        print(r.stdout.strip()[-3000:])
    if r.stderr and r.stderr.strip():
        print(r.stderr.strip()[-2000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed ({r.returncode}): {cmd}")
    return r


Path(REPO_DIR).parent.mkdir(parents=True, exist_ok=True)

# --- clone (or update) the emotion-adapter repository ------------------------
if Path(REPO_DIR, ".git").exists():
    print(f"[clone] {REPO_DIR} exists -> trying fast-forward update")
    sh("git pull --ff-only", cwd=REPO_DIR, check=False)
else:
    sh(f"git clone {REPO_URL} {REPO_DIR}")

# --- make sure the pipeline code is present ----------------------------------
# The emotion-adapter repo is where the updated code lives. If it does not
# (yet) contain the speech2face sources, bootstrap them from the base repo so
# this notebook is fully self-contained and runnable right now.
if not Path(REPO_DIR, "src", "imtalker", "generator", "FM.py").exists():
    print("[bootstrap] emotion-adapter repo has no pipeline code yet")
    print("[bootstrap] fetching the speech2face base sources ...")
    tmp = tempfile.mkdtemp(prefix="s2f_base_")
    try:
        sh(f"git clone --depth 1 {BASE_REPO_URL} {tmp}")
        src = Path(tmp) / "src"
        dst = Path(REPO_DIR) / "src"
        if src.exists() and not dst.exists():
            shutil.copytree(src, dst)
            print(f"[bootstrap] copied {src} -> {dst}")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

fm = Path(REPO_DIR, "src", "imtalker", "generator", "FM.py")
assert fm.exists(), f"Pipeline code missing after clone/bootstrap: {fm}"
print()
print(f"[ready] repository at {REPO_DIR}")
print(f"[ready] generator code at {fm}")


## Configuration

Edit the paths and settings below to match your setup. Everything else in this
notebook derives from this cell. Re-run this cell if you change anything.


In [ ]:
# ============================================================================
# CONFIGURATION -- EDIT THE PATHS AND SETTINGS BELOW
# ============================================================================
import os
from pathlib import Path

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

# Repository root (src/imtalker -- the folder that contains generator/, renderer/)
REPO_ROOT = Path("/workspace/speech2face_emotion/src/imtalker")
PROJECT_ROOT = REPO_ROOT.parent.parent          # /workspace/speech2face_emotion

# Your emotion dataset root (flat layout: motion/ audio/ smirk/ gaze/)
# Same preprocessed format as the old pipeline's emotion dataset.
EMOTION_DATASET_DIR = Path(os.environ.get("EMOTION_DATASET_DIR", "/workspace/emotion_dataset"))

# Checkpoints directory for the base IMTalker models (renderer, wav2vec2, generator)
CKPT_DIR = REPO_ROOT / "checkpoints"
WAV2VEC_DIR = CKPT_DIR / "wav2vec2-base-960h"
RENDERER_CKPT = CKPT_DIR / "renderer.ckpt"

# 1. Base generator checkpoint (the full ~38.8M parameter model from cbsjtu01/IMTalker).
#    MUST contain the core base weights (fmt.blocks.*, ...).
BASE_GENERATOR_CKPT = CKPT_DIR / "generator.ckpt"

# 2. Optional style LoRA checkpoint (e.g. the old pipeline's live-winner blink LoRA).
#    If provided, its LoRA layers are merged into the base generator BEFORE the
#    emotion LoRA is applied (same as the old training notebook).
#    None = train on the plain base generator (matches how the new pipeline's
#    FM-generator paths deploy -- they do not stack a style LoRA by default).
STYLE_LORA_CKPT = None
STYLE_LORA_RANK = 64
STYLE_LORA_ALPHA = 128.0

# Where to store training outputs (logs, checkpoints, test videos)
EXP_ROOT = PROJECT_ROOT / "emotion_exps"
EXP_NAME = "emotion_lora_v1"

# ---------------------------------------------------------------------------
# LoRA hyper-parameters (for emotion conditioning)
# ---------------------------------------------------------------------------
LORA_RANK = 32
LORA_ALPHA = 64.0
LORA_DROPOUT = 0.05

# ---------------------------------------------------------------------------
# Training hyper-parameters
# ---------------------------------------------------------------------------
BATCH_SIZE = None        # None = auto from GPU VRAM (GPU cell below)
LR = 1e-4
MAX_STEPS = 100_000
MAX_EPOCHS = -1
SAVE_FREQ = 5_000
DISPLAY_FREQ = 2_000
LOG_EVERY = 50
NUM_WORKERS = 4
VAL_SPLIT = 200

# ---------------------------------------------------------------------------
# Resume / restart
# ---------------------------------------------------------------------------
AUTO_RESUME = True
RESUME_CKPT = None

# ---------------------------------------------------------------------------
# Emotion filter (optional): None = train on ALL 6 emotions
#   0: neutral, 1: happy, 2: sad, 3: angry, 4: surprise, 5: fear
# ---------------------------------------------------------------------------
EMOTION_FILTER = None

# ---------------------------------------------------------------------------
# Mixed precision / CUDA
# ---------------------------------------------------------------------------
# "auto"     -> bf16-mixed on small-VRAM Ampere+ GPUs, fp32 otherwise
# "32-true" | "bf16-mixed" | "16-mixed"
MIXED_PRECISION = "auto"
TF32 = True

# ---------------------------------------------------------------------------
# Periodic test video configuration (runs every TEST_EVERY_N epochs)
# ---------------------------------------------------------------------------
TEST_REF_IMAGE = None    # None = auto-resolve 3robert.jpeg (see downloads cell)
TEST_AUDIO = None        # None = auto-resolve audio_3.wav
TEST_EVERY_N = 10        # <-- test the model every 10 epochs
TEST_CROP = False        # False = 512x512 resize without face_alignment crop (robot face)
TEST_EMOTION_ID = -1     # -1 = auto-cycle happy/sad/angry/surprise/fear/neutral per test
TEST_CFG_SCALE = 3.0
TEST_NFE = 10
TEST_SEED = 42

# ---------------------------------------------------------------------------
# Quick sanity prints
# ---------------------------------------------------------------------------
EXP_DIR = EXP_ROOT / EXP_NAME
print(f"repo:                {REPO_ROOT}")
print(f"project root:        {PROJECT_ROOT}")
print(f"emotion dataset:     {EMOTION_DATASET_DIR}")
print(f"base generator:      {BASE_GENERATOR_CKPT}")
print(f"style lora to merge: {STYLE_LORA_CKPT}")
print(f"renderer ckpt:       {RENDERER_CKPT}")
print(f"exp output:          {EXP_DIR}")
print(f"LoRA rank={LORA_RANK} alpha={LORA_ALPHA} | batch={BATCH_SIZE} lr={LR} max_steps={MAX_STEPS:,}")
print(f"test every {TEST_EVERY_N} epochs | emotion={TEST_EMOTION_ID} | crop={TEST_CROP}")


In [ ]:
# ============================================================================
# GPU DETECTION, CUDA CONFIGURATION, MEMORY MANAGEMENT, DEPENDENCIES
# ============================================================================
import importlib.util
import os
import shutil
import subprocess
import sys

# --- 1. torch / CUDA ---------------------------------------------------------
try:
    import torch
except ImportError:
    print("[deps] torch not found -> installing CUDA 12.8 build ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0",
                    "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU visible to torch. Use a RunPod GPU pod (PyTorch / CUDA image)."
    )

# --- 2. GPU inventory --------------------------------------------------------
GPU_COUNT = torch.cuda.device_count()
GPUS = []
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    cc = (props.major, props.minor)
    GPUS.append({
        "index": i, "name": props.name, "vram_gb": props.total_memory / 2**30,
        "cc": cc, "bf16": cc[0] >= 8,
    })
VRAM_GB = max(g["vram_gb"] for g in GPUS)
print(f"torch {torch.__version__} (CUDA {torch.version.cuda}) | {GPU_COUNT} GPU(s):")
for g in GPUS:
    print(f"  GPU[{g['index']}] {g['name']}  {g['vram_gb']:.1f} GB  sm_{g['cc'][0]}{g['cc'][1]}  bf16={g['bf16']}")

# --- 3. VRAM-based batch size ------------------------------------------------
if BATCH_SIZE is None:
    if VRAM_GB >= 70:   BATCH_SIZE = 18
    elif VRAM_GB >= 40: BATCH_SIZE = 16
    elif VRAM_GB >= 32: BATCH_SIZE = 12
    elif VRAM_GB >= 24: BATCH_SIZE = 8
    elif VRAM_GB >= 16: BATCH_SIZE = 4
    else:               BATCH_SIZE = 2
print(f"\nBATCH_SIZE (per GPU) = {BATCH_SIZE}")

# --- 4. Mixed precision ------------------------------------------------------
if MIXED_PRECISION == "auto":
    bf16_ok = all(g["bf16"] for g in GPUS)
    MIXED_PRECISION = "bf16-mixed" if (bf16_ok and VRAM_GB < 24) else "32-true"
print(f"MIXED_PRECISION = {MIXED_PRECISION}   (old pipeline default was 32-true)")

# --- 5. CUDA / memory configuration ------------------------------------------
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
if TF32:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.cuda.empty_cache()
print(f"TF32 = {TF32} | cudnn.benchmark = True | expandable_segments allocator")

# --- 6. Required packages (safe to re-run) -----------------------------------
def _pip(*pkgs):
    print("[install]", " ".join(pkgs))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + list(pkgs), check=True)

required = {
    "pytorch_lightning": "pytorch-lightning==2.2.1",
    "torchdiffeq":       "torchdiffeq==0.2.5",
    "einops":            "einops",
    "librosa":           "librosa",
    "face_alignment":    "face_alignment==1.4.1",
    "av":                "av==12.0.0",
    "tensorboard":       "tensorboard",
    "timm":              "timm==1.0.9",
    "tqdm":              "tqdm",
    "huggingface_hub":   "huggingface_hub",
    "cv2":               "opencv-python-headless",
    "PIL":               "Pillow",
    "numpy":             "numpy",
    "scipy":             "scipy",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    _pip(*missing)
else:
    print("All required packages already installed")

# transformers: needs >=4.38,<4.42 for the Wav2Vec2 eager-attention path
try:
    import transformers as _tf
    _tv = tuple(int(x) for x in _tf.__version__.split(".")[:2])
    _tf_ok = (4, 38) <= _tv < (4, 42)
    print(f"transformers {_tf.__version__} -> {'OK' if _tf_ok else 'OUT OF RANGE'}")
except Exception:
    _tf_ok = False
if not _tf_ok:
    _pip("transformers>=4.38,<4.42")

import pytorch_lightning as pl
print(f"\npytorch-lightning: {pl.__version__}")
assert shutil.which("ffmpeg"), "ffmpeg not found on PATH (needed for test videos)"
print("ffmpeg: OK")


## Install the emotion adapter code into the cloned repo

The new pipeline generator is identical to the old one except that every trace of
emotion support was removed. The cells below re-add exactly the old pipeline emotion
implementation, adapted to the new layout:

1. **`FM.py`** -- adds `EMOTION_MAP` / `EMOTION_NAMES` / `EmotionConditioner` and threads
   `emotion_id` through `forward()` (training) and `sample()` (inference, incl. the null
   emotion bias used for classifier-free guidance).
2. **`FMT.py`** -- the emotion bias is added to the conditioning sum (`a + pose + cam + gaze`)
   before `c_embedder`, and `forward_with_cfg()` gets separate conditional/unconditional
   emotion biases.
3. **`train_lora.py`** -- appends `apply_emotion_lora_to_model()` (freeze all, unfreeze the
   emotion conditioner, LoRA on `fmt.c_embedder`, all `fmt.blocks.*.adaLN_modulation.1`,
   `fmt.decoder.*` -- audio/pose/gaze/cam projections stay frozen to preserve lip-sync).
4. **`emotion_dataset.py`** -- emotion-aware dataset (parses the emotion label from the file
   stem, MEAD-style; 80% of emotional clips get a neutral `m_ref` so the model must use
   `emotion_id` to morph the face).
5. **`imtalker_train_utils.py`** -- `TextLogCallback` + `EpochTestVideoCallback` (renders a
   test video every N epochs with the in-memory EMA weights -- no training interruption).
6. **`run_emotion_lora_training.py`** -- the training launcher (base generator + optional
   style-LoRA merge + emotion LoRA + EMA + AdamW with 2x LR on the conditioner +
   cosine schedule + checkpointing + validation + periodic tests).
7. **`test_emotions.py`** -- renders all 6 emotions from a trained checkpoint.

All patches are **idempotent** (re-running a cell is safe) and **backward-compatible**:
without an `emotion_id` the patched generator behaves exactly like the stock one.


In [ ]:
# ============================================================================
# PATCH 1/3 -- generator/FM.py : EmotionConditioner + emotion conditioning
# (re-adds exactly the old pipeline emotion code, adapted to the new layout)
# ============================================================================
from pathlib import Path

FM_PATH = REPO_ROOT / "generator" / "FM.py"
src = FM_PATH.read_text(encoding="utf-8").replace("\r\n", "\n")

if "class EmotionConditioner" in src:
    print("[patch] FM.py already patched -- skipping")
else:
    def _insert_once(text, anchor, addition, before=False):
        cnt = text.count(anchor)
        assert cnt == 1, f"anchor not unique/found (count={cnt}): {anchor!r}"
        return text.replace(anchor, (addition + anchor) if before else (anchor + addition), 1)

    def _replace_once(text, old, new):
        cnt = text.count(old)
        assert cnt == 1, f"pattern not unique/found (count={cnt}): {old!r}"
        return text.replace(old, new, 1)

    # 1) emotion vocabulary + EmotionConditioner class (after the wav2vec2 import)
    anchor = "from generator.wav2vec2 import Wav2VecModel\n"
    emotion_module = """ZZ

EMOTION_MAP = {
    "neutral": 0, "neutrality": 0,
    "happy": 1, "happiness": 1,
    "sad": 2, "sadness": 2,
    "angry": 3, "anger": 3,
    "surprise": 4, "surprised": 4,
    "fear": 5, "fearful": 5,
}
EMOTION_NAMES = {0: "neutral", 1: "happy", 2: "sad", 3: "angry", 4: "surprise", 5: "fear"}
NUM_EMOTIONS = 6  # real emotion classes (0-5); index 6 = null CFG token


class EmotionConditioner(nn.Module):
    DC
    Maps a discrete emotion ID to a 7-d one-hot vector (using existing dim_e=7
    from base_options.py), projects it to dim_c=32, and expands frame-wise.

    At training time, 10% of samples have their emotion replaced with the null
    token (index 6 = dim_e-1) for classifier-free guidance. At inference time,
    emotion_id is a hardcoded integer (0-5) set once per conversation.

    Zero-init of the projection linear weight ensures the model starts with
    neutral (zero emotion bias) and degrades gracefully to pretrained behavior.
    DC
    def __init__(self, num_emotions: int = NUM_EMOTIONS,
                 dim_e: int = 7, dim_c: int = 32,
                 dropout_prob: float = 0.1):
        super().__init__()
        self.num_emotions = int(num_emotions)
        self.null_idx = int(dim_e) - 1   # index 6 = null token
        self.dropout_prob = float(dropout_prob)

        # Same pattern as audio_projection / pose_projection / gaze_projection
        self.emotion_projection = nn.Sequential(
            nn.Linear(int(dim_e), int(dim_c)),
            nn.LayerNorm(int(dim_c)),
            nn.SiLU(),
        )
        # Small normal init: provides immediate non-zero gradients from batch 1
        # without dead zones, while remaining subtle (std=0.02) to preserve pretrained base quality.
        nn.init.normal_(self.emotion_projection[0].weight, mean=0.0, std=0.02)
        nn.init.zeros_(self.emotion_projection[0].bias)

    def null_bias(self, B: int, T: int, device: torch.device) -> torch.Tensor:
        DCReturns the null-token embedding [B, T, dim_c] for unconditional CFG.DC
        null_ids = torch.full((B,), self.null_idx, dtype=torch.long, device=device)
        one_hot = F.one_hot(null_ids, num_classes=self.null_idx + 1).float()
        emb = self.emotion_projection(one_hot)
        return emb.unsqueeze(1).expand(-1, T, -1).contiguous()

    def forward(self, emotion_id: torch.Tensor, T: int,
                train: bool = False) -> torch.Tensor:
        DC
        Args:
            emotion_id : [B] long tensor, values in 0 .. num_emotions-1
            T          : number of time frames to expand to
            train      : if True, apply CFG dropout with self.dropout_prob
        Returns:
            [B, T, dim_c]  \u2013 additive bias for the conditioning sum
        DC
        B = emotion_id.shape[0]
        if train and self.dropout_prob > 0.0:
            null_mask = torch.rand(B, device=emotion_id.device) < self.dropout_prob
            emotion_id = torch.where(
                null_mask,
                torch.full_like(emotion_id, self.null_idx),
                emotion_id,
            )
        # One-hot to dim_e=7
        one_hot = F.one_hot(emotion_id.long(),
                            num_classes=self.null_idx + 1).float()  # [B, 7]
        emb = self.emotion_projection(one_hot)                       # [B, dim_c]
        return emb.unsqueeze(1).expand(-1, T, -1).contiguous()       # [B, T, dim_c]
ZZ"""
    emotion_module = emotion_module.replace("DC", "\"\"\"").replace("ZZ", "")
    src = _insert_once(src, anchor, emotion_module)

    # 2) emotion_conditioner module in FMGenerator.__init__
    anchor = "        self.odeint_kwargs = {"
    init_block = """PP        # Emotion conditioning (uses already-reserved dim_e and emotion_dropout_prob
        # from base_options.py; both are present but previously unused).
        self.emotion_conditioner = EmotionConditioner(
            num_emotions=NUM_EMOTIONS,
            dim_e=int(getattr(opt, "dim_e", 7)),
            dim_c=opt.dim_c,
            dropout_prob=float(getattr(opt, "emotion_dropout_prob", 0.1)),
        )

PP"""
    init_block = init_block.replace("PP", "")
    src = _insert_once(src, anchor, init_block, before=True)

    # 3) forward(): read emotion_id from the batch
    anchor = "        bs = x.size(0)\n"
    src = _insert_once(src, anchor, "\n        emotion_id = batch.get(\"emotion_id\", None)\n")

    # 4) forward(): compute the per-frame emotion bias (before the fmt call)
    anchor = "        pred = self.fmt(\n"
    bias_block = """PP        # Compute per-frame emotion bias in dim_c space
        if emotion_id is not None:
            T_clip = a.shape[1]   # frames after audio projection
            emotion_bias_now  = self.emotion_conditioner(emotion_id, T_clip, train=self.training)
            emotion_bias_prev = self.emotion_conditioner(emotion_id, self.num_prev_frames, train=self.training)
        else:
            emotion_bias_now  = None
            emotion_bias_prev = None

PP"""
    bias_block = bias_block.replace("PP", "")
    src = _insert_once(src, anchor, bias_block, before=True)

    # 5) forward(): pass the biases to the FMT
    src = _replace_once(
        src,
        "            train=self.training,\n        )",
        "            train=self.training,\n"
        "            emotion_bias_now=emotion_bias_now,\n"
        "            emotion_bias_prev=emotion_bias_prev,\n"
        "        )",
    )

    # 6) sample(): resolve the emotion id tensor
    sample_anchor = "        cam = self._project_condition_or_zero(cam, self.cam_projection, B, T)\n\n        sample = []"
    sample_block = """PP        cam = self._project_condition_or_zero(cam, self.cam_projection, B, T)

        # Emotion conditioning for sample()
        _emotion_id_raw = data.get("emotion_id", None)
        if _emotion_id_raw is not None:
            if isinstance(_emotion_id_raw, int):
                _eid_tensor = torch.tensor([_emotion_id_raw], dtype=torch.long, device=device)
            else:
                _eid_tensor = torch.as_tensor(_emotion_id_raw, dtype=torch.long, device=device)
                if _eid_tensor.ndim == 0:
                    _eid_tensor = _eid_tensor.unsqueeze(0)
            _eid_tensor = _eid_tensor.expand(B)
        else:
            _eid_tensor = None

        sample = []PP"""
    sample_block = sample_block.replace("PP", "")
    src = _replace_once(src, sample_anchor, sample_block)

    # 7) sample(): per-chunk emotion bias (+ null bias for CFG)
    anchor = "            def sample_chunk(tt, zt):"
    chunk_block = """PP            # Compute emotion bias for this chunk (and null bias for unconditional CFG)
            if _eid_tensor is not None:
                _ebias_now  = self.emotion_conditioner(_eid_tensor, self.num_frames_for_clip, train=False)
                _ebias_prev = self.emotion_conditioner(_eid_tensor, self.num_prev_frames, train=False)
                _null_ebias_now  = self.emotion_conditioner.null_bias(B, self.num_frames_for_clip, device=device)
                _null_ebias_prev = self.emotion_conditioner.null_bias(B, self.num_prev_frames, device=device)
            else:
                _ebias_now  = None
                _ebias_prev = None
                _null_ebias_now  = None
                _null_ebias_prev = None

PP"""
    chunk_block = chunk_block.replace("PP", "")
    src = _insert_once(src, anchor, chunk_block, before=True)

    # 8) sample(): pass emotion biases to forward_with_cfg
    src = _replace_once(
        src,
        "                    a_cfg_scale=a_cfg_scale,\n                )",
        "                    a_cfg_scale=a_cfg_scale,\n"
        "                    emotion_bias_now=_ebias_now,\n"
        "                    emotion_bias_prev=_ebias_prev,\n"
        "                    null_emotion_bias_now=_null_ebias_now,\n"
        "                    null_emotion_bias_prev=_null_ebias_prev,\n"
        "                )",
    )

    FM_PATH.write_text(src, encoding="utf-8")
    compile(src, str(FM_PATH), "exec")   # syntax check of the patched file
    print("[patch] FM.py patched: EmotionConditioner + emotion conditioning in forward()/sample()")


In [ ]:
# ============================================================================
# PATCH 2/3 -- generator/FMT.py : emotion bias in the conditioning sum + CFG
# (re-adds exactly the old pipeline emotion code)
# ============================================================================
from pathlib import Path

FMT_PATH = REPO_ROOT / "generator" / "FMT.py"
src = FMT_PATH.read_text(encoding="utf-8").replace("\r\n", "\n")

if "emotion_bias_now" in src:
    print("[patch] FMT.py already patched -- skipping")
else:
    def _replace_once(text, old, new):
        cnt = text.count(old)
        assert cnt == 1, f"pattern not unique/found (count={cnt}): {old!r}"
        return text.replace(old, new, 1)

    # 1) forward() signature: accept the emotion biases
    src = _replace_once(
        src,
        "        train: bool = True,\n        **kwargs\n    ) -> torch.Tensor:",
        "        train: bool = True,\n"
        "        emotion_bias_now=None,\n"
        "        emotion_bias_prev=None,\n"
        "        **kwargs\n"
        "    ) -> torch.Tensor:",
    )

    # 2) forward(): add the emotion bias to the conditioning sum (before c_embedder)
    old_cond = "        c = self.c_embedder(a + pose + cam + gaze)\n        c = t + c"
    new_cond = """PP        # Build conditioning sum; add emotion bias BEFORE c_embedder projection
        conditioning_sum = a + pose + cam + gaze
        if emotion_bias_now is not None:
            T_total = conditioning_sum.shape[1]
            # Build full-sequence emotion bias
            if emotion_bias_prev is not None:
                full_emotion_bias = torch.cat([emotion_bias_prev, emotion_bias_now], dim=1)
            else:
                full_emotion_bias = emotion_bias_now
            # Trim/pad to match actual sequence length
            if full_emotion_bias.shape[1] != T_total:
                full_emotion_bias = F.interpolate(
                    full_emotion_bias.transpose(1, 2),
                    size=T_total, mode='nearest'
                ).transpose(1, 2)
            conditioning_sum = conditioning_sum + full_emotion_bias
        c = self.c_embedder(conditioning_sum)
        c = t + cPP""".replace("PP", "")
    src = _replace_once(src, old_cond, new_cond)

    # 3) forward_with_cfg() signature: accept conditional + null emotion biases
    src = _replace_once(
        src,
        "        a_cfg_scale: float = 1.0,\n        **kwargs\n    ) -> torch.Tensor:",
        "        a_cfg_scale: float = 1.0,\n"
        "        emotion_bias_now=None,\n"
        "        emotion_bias_prev=None,\n"
        "        null_emotion_bias_now=None,\n"
        "        null_emotion_bias_prev=None,\n"
        "        **kwargs\n"
        "    ) -> torch.Tensor:",
    )

    # 4) forward_with_cfg(): build the uncond/cond emotion bias batches
    old_ref = "            ref_x_cat     = torch.cat([ref_x, ref_x], dim=0)\n\n            model_output = self.forward("
    new_ref = """PP            ref_x_cat     = torch.cat([ref_x, ref_x], dim=0)

            # Emotion bias: uncond branch uses null token representation; cond branch uses target emotion
            if emotion_bias_now is not None:
                uncond_now = null_emotion_bias_now if null_emotion_bias_now is not None else emotion_bias_now
                ebias_now_cat = torch.cat([uncond_now, emotion_bias_now], dim=0)
            else:
                ebias_now_cat = None

            if emotion_bias_prev is not None:
                uncond_prev = null_emotion_bias_prev if null_emotion_bias_prev is not None else emotion_bias_prev
                ebias_prev_cat = torch.cat([uncond_prev, emotion_bias_prev], dim=0)
            else:
                ebias_prev_cat = None

            model_output = self.forward(PP""".replace("PP", "")
    src = _replace_once(src, old_ref, new_ref)

    # 5) first forward() call (CFG branch): pass the emotion biases
    src = _replace_once(
        src,
        "                prev_cam=prev_cam_cat,\n                train=False\n            )",
        "                prev_cam=prev_cam_cat,\n"
        "                train=False,\n"
        "                emotion_bias_now=ebias_now_cat,\n"
        "                emotion_bias_prev=ebias_prev_cat,\n"
        "            )",
    )

    # 6) second forward() call (no-CFG branch): pass the emotion biases
    src = _replace_once(
        src,
        "                prev_cam=prev_cam,\n                train=False\n            )",
        "                prev_cam=prev_cam,\n"
        "                train=False,\n"
        "                emotion_bias_now=emotion_bias_now,\n"
        "                emotion_bias_prev=emotion_bias_prev,\n"
        "            )",
    )

    FMT_PATH.write_text(src, encoding="utf-8")
    compile(src, str(FMT_PATH), "exec")
    print("[patch] FMT.py patched: emotion bias in conditioning sum + CFG branches")


In [ ]:
# ============================================================================
# PATCH 3/3 -- generator/train_lora.py : append apply_emotion_lora_to_model()
# (exactly the old pipeline function)
# ============================================================================
from pathlib import Path

TL_PATH = REPO_ROOT / "generator" / "train_lora.py"
src = TL_PATH.read_text(encoding="utf-8").replace("\r\n", "\n")

if "def apply_emotion_lora_to_model" in src:
    print("[patch] train_lora.py already patched -- skipping")
else:
    block = """PP


def apply_emotion_lora_to_model(
    model,
    rank: int = 32,
    alpha: float = None,
    dropout: float = 0.05,
):
    DC
    Emotion-specific LoRA fine-tuning setup.

    Freezes ALL base model parameters, then:
    1. Unfreezes the emotion_conditioner entirely (new, small module).
    2. Applies LoRA to: fmt.c_embedder, all fmt.blocks.*.adaLN_modulation.1,
       fmt.decoder.linear, fmt.decoder.adaLN_modulation.1.

    These are exactly the layers that control how conditioning signals
    (now including emotion) modulate each transformer block.
    Audio/pose/gaze/cam projections are frozen to preserve lip-sync quality.
    DC
    if alpha is None or float(alpha) <= 0:
        alpha = float(2 * int(rank))

    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze emotion_conditioner (new module, not in pretrained ckpt)
    for param in model.emotion_conditioner.parameters():
        param.requires_grad = True

    # Collect LoRA targets across transformer blocks, conditioning embedder, and decoder
    targets = []
    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue
        is_attn_mlp = name.startswith("fmt.blocks.") and (
            name.endswith(".attn.qkv")
            or name.endswith(".attn.proj")
            or name.endswith(".mlp.fc1")
            or name.endswith(".mlp.fc2")
        )
        is_adaLN = (
            name.startswith("fmt.blocks.")
            and name.endswith(".adaLN_modulation.1")
        )
        is_decoder = name in {
            "fmt.decoder.linear",
            "fmt.decoder.adaLN_modulation.1",
        }
        is_c_embedder = name == "fmt.c_embedder"
        if is_attn_mlp or is_adaLN or is_decoder or is_c_embedder:
            targets.append(name)

    for name in targets:
        base = _get_child(model, name)
        _set_child(
            model, name,
            LoRALinear(base, rank=int(rank), alpha=float(alpha), dropout=float(dropout))
        )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(
        f"[EmotionLoRA] targets={len(targets)} rank={rank} alpha={alpha} dropout={dropout}",
        flush=True,
    )
    print(
        f"[EmotionLoRA] trainable={trainable:,} total={total:,} "
        f"ratio={trainable / max(1, total):.6f}",
        flush=True,
    )
    return targets
PP""".replace("PP", "").replace("DC", "\"\"\"")
    src = src.rstrip("\n") + block
    TL_PATH.write_text(src, encoding="utf-8")
    compile(src, str(TL_PATH), "exec")
    print("[patch] train_lora.py patched: apply_emotion_lora_to_model() appended")


In [ ]:
# ============================================================================
# CREATE generator/emotion_dataset.py (identical to the old pipeline file)
# ============================================================================
from pathlib import Path

_SRC = r'''
"""
Emotion-aware dataset for IMTalker emotion LoRA fine-tuning.

Drop-in replacement for generator/dataset.py's AudioMotionSmirkGazeDataset
that additionally:
  - infers the emotion label from the preprocessed file stem
  - returns an "emotion_id" key (long tensor) in every batch dict

Filename parsing rules (derived from user's MEAD dataset naming):
  neutral_XXXX.mp4          -> "neutral"  (index 0)
  M003_happy_level_1_001.mp4 -> "happy"   (index 1)
  M003_sad_level_2_018.mp4   -> "sad"     (index 2)
  M011_angry_level_3_030.mp4 -> "angry"   (index 3)
  M022_surprise_level_2_016  -> "surprise"(index 4)
  M009_fear_level_1_016.mp4  -> "fear"    (index 5)

The parser splits the stem on '_' and checks each token against EMOTION_MAP.
Unknown filenames default to neutral (0) with a printed warning.
"""
from __future__ import annotations

import random
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from torch.utils.data import Dataset
from tqdm import tqdm

from generator.dataset import load_pose

# -- Emotion vocabulary ------------------------------------------------------
EMOTION_MAP: dict[str, int] = {
    "neutral":   0,
    "neutrality":0,
    "happy":     1,
    "happiness": 1,
    "sad":       2,
    "sadness":   2,
    "angry":     3,
    "anger":     3,
    "surprise":  4,
    "surprised": 4,
    "fear":      5,
    "fearful":   5,
}
EMOTION_NAMES: dict[int, str] = {
    0: "neutral",
    1: "happy",
    2: "sad",
    3: "angry",
    4: "surprise",
    5: "fear",
}
NUM_EMOTIONS = 6          # real classes (0-5)
NULL_EMOTION_IDX = 6      # reserved for CFG dropout (dim_e-1 in 7-d one-hot)


def parse_emotion_from_stem(stem: str) -> int:
    """
    Infer emotion label from a preprocessed file stem.

    Splits on '_' and returns the EMOTION_MAP index of the first token that
    matches a known emotion name.  Falls back to 0 (neutral) with a warning.

    Examples
    --------
    >>> parse_emotion_from_stem("neutral_0001")          # -> 0
    >>> parse_emotion_from_stem("M003_happy_level_1_001") # -> 1
    >>> parse_emotion_from_stem("M009_fear_level_1_016")  # -> 5
    """
    tokens = stem.lower().split("_")
    for token in tokens:
        if token in EMOTION_MAP:
            return EMOTION_MAP[token]
    warnings.warn(
        f"[EmotionDataset] Could not infer emotion from stem '{stem}'; "
        "defaulting to neutral (0). Check EMOTION_MAP if this is unexpected.",
        stacklevel=2,
    )
    return 0  # default: neutral


# -- Dataset -----------------------------------------------------------------

class EmotionAudioMotionDataset(Dataset):
    """
    Emotion-aware dataset for emotion LoRA fine-tuning.

    Identical to AudioMotionSmirkGazeDataset but:
      1. Each sample dict contains an additional "emotion_id" key (long tensor
         scalar, value in 0..NUM_EMOTIONS-1).
      2. Can be filtered to only include specific emotion(s) via
         ``emotion_filter``.
      3. Prints a per-emotion breakdown during initialisation.

    Parameters
    ----------
    opt :
        The same options object used for AudioMotionSmirkGazeDataset.
    start, end :
        Slice of the sorted motion file list (same semantics as the parent).
    split :
        "train" or "val" (informational only; not used for filtering).
    list_path :
        Optional explicit list of file stems (one per line, relative to
        dataset_path).  Overrides start/end when provided.
    emotion_filter :
        If set, only samples whose inferred emotion is in this list are kept
        (e.g. ["happy", "sad"]).  None = keep all.
    """

    def __init__(
        self,
        opt,
        start: Optional[int] = None,
        end: Optional[int] = None,
        split: Optional[str] = None,
        list_path: Optional[str] = None,
        emotion_filter: Optional[list[str]] = None,
    ):
        super().__init__()
        self.opt = opt
        self.num_frames_for_clip = int(opt.wav2vec_sec * opt.fps)
        self.num_prev_frames = int(opt.num_prev_frames)
        self.required_len = self.num_frames_for_clip + self.num_prev_frames

        root_path = Path(opt.dataset_path)
        motion_dir  = root_path / "motion"
        audio_dir   = root_path / "audio"
        smirk_dir   = root_path / "smirk"
        gaze_dir    = root_path / "gaze"

        # Build file list (same logic as parent)
        if list_path and Path(list_path).exists():
            motion_files = []
            with open(list_path, "r", encoding="utf-8") as fh:
                for line in fh:
                    stem = Path(line.strip()).stem
                    if not stem:
                        continue
                    mp = motion_dir / f"{stem}.pt"
                    if mp.exists():
                        motion_files.append(mp)
        else:
            motion_files = sorted(motion_dir.glob("*.pt"))
            if start is not None or end is not None:
                motion_files = motion_files[start:end]

        # Optional emotion filter
        _filter_ids: Optional[set[int]] = None
        if emotion_filter:
            _filter_ids = set()
            for name in emotion_filter:
                name_lc = name.strip().lower()
                if name_lc not in EMOTION_MAP:
                    raise ValueError(
                        f"Unknown emotion '{name}' in emotion_filter. "
                        f"Valid: {list(EMOTION_MAP.keys())}"
                    )
                _filter_ids.add(EMOTION_MAP[name_lc])

        # Scan valid samples
        self.samples: list[dict] = []
        emotion_counts: dict[int, int] = {i: 0 for i in range(NUM_EMOTIONS)}
        unknown_count = 0

        for motion_path in tqdm(motion_files, desc=f"[EmotionDataset/{split or 'all'}] scanning"):
            stem = motion_path.stem

            audio_path = audio_dir / f"{stem}.npy"
            gaze_path  = gaze_dir  / f"{stem}.npy"
            smirk_path = smirk_dir / f"{stem}.pt"

            if not (audio_path.exists() and gaze_path.exists() and smirk_path.exists()):
                continue

            eid = parse_emotion_from_stem(stem)
            if eid not in emotion_counts:
                unknown_count += 1
                eid = 0  # fallback

            if _filter_ids is not None and eid not in _filter_ids:
                continue

            try:
                motion = torch.load(str(motion_path), map_location="cpu")
                smirk  = torch.load(str(smirk_path),  map_location="cpu")
                audio  = np.load(str(audio_path), mmap_mode="r")
                gaze   = np.load(str(gaze_path),  mmap_mode="r")

                min_len = min(
                    len(motion),
                    len(audio),
                    len(gaze),
                    len(smirk["pose_params"]),
                )
                if min_len < self.required_len:
                    continue

                self.samples.append({
                    "motion_path": str(motion_path),
                    "audio_path":  str(audio_path),
                    "smirk_path":  str(smirk_path),
                    "gaze_path":   str(gaze_path),
                    "emotion_id":  eid,
                    "stem":        stem,
                })
                emotion_counts[eid] = emotion_counts.get(eid, 0) + 1

            except Exception as exc:
                print(f"[EmotionDataset] skip {stem}: {exc}")
                continue

        if not self.samples:
            raise RuntimeError(
                f"No valid samples found in {root_path}. "
                "Run preprocessing notebooks first."
            )

        print(f"[EmotionDataset/{split or 'all'}] {len(self.samples)} valid samples")
        for eid, cnt in sorted(emotion_counts.items()):
            if cnt > 0:
                print(f"  {EMOTION_NAMES[eid]:10s} (id={eid}): {cnt}")
        if unknown_count:
            print(f"  [unknown->neutral]: {unknown_count}")

        # Pre-cache a pool of neutral reference frames (m_ref)
        # When training on emotion clips (happy, sad, etc.), using a neutral m_ref forces
        # the model to learn to transform a neutral resting face into the target emotion.
        self.neutral_indices = [i for i, s in enumerate(self.samples) if s["emotion_id"] == 0]
        self.neutral_ref_pool = []
        if self.neutral_indices:
            n_pool_samples = min(50, len(self.neutral_indices))
            for s_idx in self.neutral_indices[:n_pool_samples]:
                try:
                    m_data = torch.load(self.samples[s_idx]["motion_path"], map_location="cpu")
                    if m_data.ndim == 2 and m_data.shape[0] > 0:
                        step = max(1, m_data.shape[0] // 5)
                        for f_idx in range(0, m_data.shape[0], step):
                            self.neutral_ref_pool.append(m_data[f_idx].clone().float())
                except Exception:
                    pass
            if self.neutral_ref_pool:
                self.neutral_ref_pool = torch.stack(self.neutral_ref_pool)  # [N_neutral, 32]
                print(f"[EmotionDataset/{split or 'all'}] Cached {len(self.neutral_ref_pool)} neutral reference frames for expression transformation")

    def __len__(self) -> int:
        return len(self.samples)

    def _get_full_clip(self, index: int):
        item = self.samples[index]

        motion  = torch.load(item["motion_path"], map_location="cpu")
        audio   = np.load(item["audio_path"], mmap_mode="r")
        gaze    = np.load(item["gaze_path"],  mmap_mode="r")
        smirk   = torch.load(item["smirk_path"], map_location="cpu")
        pose, cam = load_pose(smirk)

        min_len = min(len(motion), len(audio), len(gaze), len(pose))
        start_idx = random.randint(0, min_len - self.required_len)
        end_idx   = start_idx + self.required_len

        audio_seg  = torch.from_numpy(audio[start_idx:end_idx].copy()).float()
        motion_seg = motion[start_idx:end_idx]
        gaze_seg   = torch.from_numpy(gaze[start_idx:end_idx].copy()).float()
        pose_seg   = pose[start_idx:end_idx]
        cam_seg    = cam[start_idx:end_idx]

        np_ = self.num_prev_frames
        return (
            motion_seg[np_:],  audio_seg[np_:],  # clip
            motion_seg[:np_],  audio_seg[:np_],  # prev
            motion_seg,                           # full (for ref)
            gaze_seg[np_:],   gaze_seg[:np_],
            pose_seg[np_:],   pose_seg[:np_],
            cam_seg[np_:],    cam_seg[:np_],
        )

    def __getitem__(self, index: int) -> dict:
        try:
            (mc, ac, mp_, ap_, ms,
             gc, gp, pc, pp, cc, cp) = self._get_full_clip(index)
        except Exception as exc:
            print(f"[EmotionDataset] error idx={index}: {exc}; retrying random")
            return self.__getitem__(random.randint(0, len(self) - 1))

        emotion_id = self.samples[index]["emotion_id"]

        # Expression-morphing reference selection:
        # If the sample is an emotional clip (happy, sad, etc.), with 80% probability
        # pick m_ref from the NEUTRAL reference pool.
        # This breaks the shortcut where m_ref already contains the target emotion,
        # forcing the model to actively use emotion_id to morph the neutral face into
        # the requested emotion.
        if (
            emotion_id != 0
            and hasattr(self, "neutral_ref_pool")
            and isinstance(self.neutral_ref_pool, torch.Tensor)
            and len(self.neutral_ref_pool) > 0
            and random.random() < 0.8
        ):
            n_idx = random.randint(0, len(self.neutral_ref_pool) - 1)
            m_ref = self.neutral_ref_pool[n_idx]
        else:
            ref_idx = torch.randint(0, ms.shape[0], (1,)).item()
            m_ref = ms[ref_idx]

        return {
            # motion / audio
            "m_now":   mc,  "a_now":   ac,
            "m_prev":  mp_, "a_prev":  ap_,
            "m_ref":   m_ref,
            # gaze
            "gaze_now": gc, "gaze_prev": gp,
            # pose / cam (smirk)
            "pose_now": pc, "pose_prev": pp,
            "cam_now":  cc, "cam_prev":  cp,
            # emotion label
            "emotion_id": torch.tensor(emotion_id, dtype=torch.long),
        }
'''

_dst = REPO_ROOT / "generator/emotion_dataset.py"
_dst.write_text(_SRC.lstrip("\n"), encoding="utf-8")
compile(_SRC, str(_dst), "exec")
print(f"[write] {_dst} ({len(_SRC.splitlines())} lines)")


In [ ]:
# ============================================================================
# CREATE generator/imtalker_train_utils.py (identical to the old pipeline file)
# ============================================================================
from pathlib import Path

_SRC = r'''
"""
Training utilities for IMTalker Generator and Emotion LoRA training.

Contains two Lightning callbacks:
  - TextLogCallback:        appends human-readable progress (epoch / step /
                            loss / lr) to a plain-text log file on rank 0.
  - EpochTestVideoCallback: after every N training epochs, renders a test video
                            using the live in-memory generator weights and
                            the IMTRenderer. Rank 0 only.
"""
import math
import os
import time
from pathlib import Path

import torch
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning.utilities import rank_zero_only


class TextLogCallback(pl.Callback):
    """Writes epoch / step / loss / lr lines to a plain-text log file (rank 0)."""

    def __init__(self, log_file, every_n_steps=50):
        super().__init__()
        self.log_file = Path(log_file)
        self.every_n_steps = max(1, int(every_n_steps))
        self.log_file.parent.mkdir(parents=True, exist_ok=True)

    def _write(self, line):
        with open(self.log_file, "a", encoding="utf-8") as f:
            f.write(line + "\n")

    @rank_zero_only
    def on_train_start(self, trainer, pl_module):
        self._write(
            f"\n[{time.strftime('%Y-%m-%d %H:%M:%S')}] training started | "
            f"gpus={trainer.num_devices} | max_steps={trainer.max_steps} | "
            f"max_epochs={trainer.max_epochs}"
        )

    @rank_zero_only
    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if trainer.global_step % self.every_n_steps != 0:
            return
        lr = trainer.optimizers[0].param_groups[0]["lr"]
        loss = float(trainer.callback_metrics.get("train_loss", float("nan")))
        fm = float(trainer.callback_metrics.get("fm_loss", float("nan")))
        self._write(
            f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] epoch={trainer.current_epoch:04d} "
            f"step={trainer.global_step:07d} lr={lr:.3e} "
            f"train_loss={loss:.6f} fm_loss={fm:.6f}"
        )

    @rank_zero_only
    def on_validation_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        m = trainer.callback_metrics
        self._write(
            f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] VALIDATION epoch={trainer.current_epoch:04d} "
            f"step={trainer.global_step:07d} "
            f"val_loss={float(m.get('val_loss', float('nan'))):.6f} "
            f"val_fm_loss={float(m.get('val_fm_loss', float('nan'))):.6f}"
        )

    @rank_zero_only
    def on_train_epoch_end(self, trainer, pl_module):
        self._write(
            f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] epoch={trainer.current_epoch:04d} "
            f"completed at step={trainer.global_step:07d}"
        )


class EpochTestVideoCallback(pl.Callback):
    """
    Renders a test video after every N training epochs directly with the current
    generator weights (pl_module.model) and IMTRenderer. Rank 0 only.
    """

    def __init__(self, ref_path, aud_path, out_dir, renderer_ckpt, wav2vec_model_path,
                 every_n_epochs=1, crop=False, a_cfg_scale=3.0, nfe=10, seed=42, test_emotion_id=1):
        super().__init__()
        self.ref_path = str(ref_path) if ref_path else None
        self.aud_path = str(aud_path) if aud_path else None
        self.out_dir = Path(out_dir)
        self.renderer_ckpt = str(renderer_ckpt)
        self.wav2vec_model_path = str(wav2vec_model_path)
        self.every_n_epochs = max(1, int(every_n_epochs))
        self.crop = bool(crop)
        self.a_cfg_scale = float(a_cfg_scale)
        self.nfe = int(nfe)
        self.seed = int(seed)
        self.test_emotion_id = int(test_emotion_id)
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.renderer = None
        self.wav2vec = None
        self.data_processor = None
        self.opt = None

    def _init_components(self, device):
        if self.renderer is not None:
            return

        import argparse
        from renderer.models import IMTRenderer
        from generator.generate import DataProcessor, InferenceOptions
        from generator.wav2vec2 import Wav2VecModel

        parser = argparse.ArgumentParser()
        parser = InferenceOptions().initialize(parser)
        opt = parser.parse_args([])
        opt.rank = device
        opt.renderer_path = self.renderer_ckpt
        opt.wav2vec_model_path = self.wav2vec_model_path
        self.opt = opt

        # 1. Renderer
        self.renderer = IMTRenderer(opt).to(device).eval()
        r_ckpt = torch.load(self.renderer_ckpt, map_location="cpu", weights_only=False)
        r_state = r_ckpt.get("state_dict", r_ckpt)
        ae_state = {k.replace("gen.", ""): v for k, v in r_state.items() if k.startswith("gen.")}
        self.renderer.load_state_dict(ae_state, strict=False)
        print(f"[EpochTest] Renderer loaded from {Path(self.renderer_ckpt).name}", flush=True)

        # 2. Wav2Vec for audio feature extraction
        self.wav2vec = Wav2VecModel.from_pretrained(self.wav2vec_model_path, local_files_only=True).to(device).eval()
        self.wav2vec.feature_extractor._freeze_parameters()
        for p in self.wav2vec.parameters():
            p.requires_grad = False
        print(f"[EpochTest] Wav2Vec2 loaded from {Path(self.wav2vec_model_path).name}", flush=True)

        # 3. DataProcessor
        self.data_processor = DataProcessor(opt)

    def _extract_audio_features(self, speech_tensor, device, fps=25, sr=16000):
        # speech_tensor is [1, samples]
        raw = speech_tensor.squeeze(0).to(device)
        total_samples = raw.shape[0]
        T = int(math.ceil(total_samples * fps / sr))
        target_samples = int(T * sr / fps)
        if raw.shape[0] < target_samples:
            diff = target_samples - raw.shape[0]
            raw = F.pad(raw.unsqueeze(0), (0, diff), mode="replicate").squeeze(0)
        else:
            raw = raw[:target_samples]
        
        with torch.no_grad():
            out = self.wav2vec(raw.unsqueeze(0), seq_len=T, output_hidden_states=False)
            feat = out.last_hidden_state  # [1, T, 768]
        return feat, T

    @rank_zero_only
    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.current_epoch % self.every_n_epochs != 0:
            return
        if not self.ref_path or not Path(self.ref_path).exists():
            return
        if not self.aud_path or not Path(self.aud_path).exists():
            return

        try:
            device = pl_module.device
            self._init_components(device)

            # Apply EMA shadow if present so test video uses clean averaged weights
            if hasattr(pl_module, "ema") and pl_module.ema is not None:
                pl_module.ema.apply_shadow()

            was_training = pl_module.model.training
            pl_module.model.eval()

            with torch.no_grad():
                data = self.data_processor.preprocess(self.ref_path, self.aud_path, crop=self.crop)
                s = data["s"].to(device)
                f_r, g_r = self.renderer.dense_feature_encoder(s)
                t_r = self.renderer.latent_token_encoder(s)
                data["ref_x"] = t_r
                data["emotion_id"] = self.test_emotion_id
                data["pose"] = None
                data["cam"] = None
                data["gaze"] = None

                # Extract audio features directly so skip_fm_audio_encoder works perfectly
                feat_25, T = self._extract_audio_features(data["a"], device)
                data["a_feat"] = feat_25

                from generator.FM import EMOTION_NAMES
                # If test_emotion_id is -1, automatically cycle through all emotions across epochs:
                # epoch 1 -> happy, epoch 2 -> sad, epoch 3 -> angry, epoch 4 -> surprise, epoch 5 -> fear, epoch 6 -> neutral
                if self.test_emotion_id < 0:
                    cycle = [1, 2, 3, 4, 5, 0]
                    current_eid = cycle[(max(0, trainer.current_epoch - 1)) % len(cycle)]
                else:
                    current_eid = int(self.test_emotion_id)

                emo_str = EMOTION_NAMES.get(current_eid, f"emotion_{current_eid}")
                data["emotion_id"] = current_eid
                print(f"[EpochTest] epoch {trainer.current_epoch}: generating test video with emotion={emo_str} (id={current_eid}) | ref={Path(self.ref_path).name} | a_cfg={self.a_cfg_scale}", flush=True)

                # Sample motion latents with the in-memory trained model
                sample = pl_module.model.sample(
                    data,
                    a_cfg_scale=self.a_cfg_scale,
                    nfe=self.nfe,
                    seed=self.seed,
                )

                # Decode frames with IMTRenderer
                T = sample.shape[1]
                ta_r = self.renderer.adapt(t_r, g_r)
                m_r = self.renderer.latent_token_decoder(ta_r)
                d_hat = []
                for t in range(T):
                    ta_c = self.renderer.adapt(sample[:, t, ...], g_r)
                    m_c = self.renderer.latent_token_decoder(ta_c)
                    d_i = self.renderer.decode(m_c, m_r, f_r)
                    d_hat.append(d_i)

                vid_tensor = torch.cat(d_hat, dim=0)  # [T, 3, 512, 512]
                out_mp4 = self.out_dir / f"epoch_{trainer.current_epoch:04d}_{emo_str}.mp4"
                self._save_video(vid_tensor, out_mp4, self.aud_path)
                # Also save canonical epoch_XXXX.mp4
                legacy_mp4 = self.out_dir / f"epoch_{trainer.current_epoch:04d}.mp4"
                if legacy_mp4 != out_mp4:
                    import shutil
                    shutil.copy2(str(out_mp4), str(legacy_mp4))
                print(f"[EpochTest] epoch {trainer.current_epoch}: successfully saved test video -> {out_mp4.name}", flush=True)

            if was_training:
                pl_module.model.train()
            if hasattr(pl_module, "ema") and pl_module.ema is not None:
                pl_module.ema.restore()

        except Exception as e:
            print(f"[EpochTest] epoch {trainer.current_epoch}: FAILED - {e}", flush=True)
            import traceback
            traceback.print_exc()

    def _save_video(self, vid_tensor, video_path, audio_path):
        import tempfile, subprocess, torchvision
        video_path = str(video_path)
        with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
            temp_filename = tmp.name

        # vid_tensor is [T, 3, 512, 512] with values in [0, 1]
        vid = vid_tensor.permute(0, 2, 3, 1).detach().clamp(0, 1).cpu()
        vid = (vid * 255).type(torch.ByteTensor)
        torchvision.io.write_video(temp_filename, vid, fps=getattr(self.opt, "fps", 25))

        if audio_path and Path(audio_path).exists():
            cmd = f'ffmpeg -i "{temp_filename}" -i "{audio_path}" -c:v copy -c:a aac -shortest "{video_path}" -y'
            subprocess.call(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            if os.path.exists(temp_filename):
                os.remove(temp_filename)
        else:
            if os.path.exists(video_path):
                os.remove(video_path)
            os.rename(temp_filename, video_path)
'''

_dst = REPO_ROOT / "generator/imtalker_train_utils.py"
_dst.write_text(_SRC.lstrip("\n"), encoding="utf-8")
compile(_SRC, str(_dst), "exec")
print(f"[write] {_dst} ({len(_SRC.splitlines())} lines)")


In [ ]:
# ============================================================================
# CREATE generator/run_emotion_lora_training.py (old pipeline file + --precision + new-layout fallback path)
# ============================================================================
from pathlib import Path

_SRC = r'''
"""
IMTalker Emotion LoRA Training -- launcher script.

Trains the emotion conditioning adapters on top of a pretrained IMTalker
generator checkpoint. The base generator is frozen; only the emotion
conditioner module and targeted LoRA adapters are trainable.

Usage (from the training notebook):
    python generator/run_emotion_lora_training.py \
        --repo_root /workspace/speech2avatar/IMTalker \
        --dataset_path /workspace/emotion_dataset \
        --pretrained_ckpt /path/to/generator.ckpt \
        --exp_root /workspace/emotion_exps \
        --exp_name emotion_lora_v1 \
        --wav2vec_model_path /path/to/wav2vec2-base-960h \
        --renderer_ckpt /path/to/renderer.ckpt \
        --lora_rank 32 \
        --lora_alpha 64.0 \
        --batch_size 8 \
        --lr 1e-4 \
        --max_steps 100000
"""
from __future__ import annotations

import argparse
import math
import os
import sys
import time
from pathlib import Path

import torch

# torch>=2.6 weights_only=True breaks PL 2.2.1 checkpoint loading
_TORCH_LOAD = torch.load
def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _TORCH_LOAD(*args, **kwargs)
torch.load = _torch_load_compat


# -- CLI ----------------------------------------------------------------------

def parse_args(argv=None):
    p = argparse.ArgumentParser(description="IMTalker emotion LoRA training")
    p.add_argument("--repo_root",           required=True)
    p.add_argument("--dataset_path",        required=True)
    p.add_argument("--base_generator_ckpt", default=None,
                   help="Path to full pretrained base generator checkpoint (generator.ckpt)")
    p.add_argument("--style_lora_ckpt",     default=None,
                   help="Optional pretrained style LoRA checkpoint (e.g. ditto_blink_lora) to merge into base")
    p.add_argument("--style_lora_rank",     type=int, default=64)
    p.add_argument("--style_lora_alpha",    type=float, default=128.0)
    p.add_argument("--pretrained_ckpt",     default=None,
                   help="Legacy alias for base_generator_ckpt or style_lora_ckpt")
    p.add_argument("--exp_root",            required=True)
    p.add_argument("--exp_name",            required=True)
    p.add_argument("--wav2vec_model_path",  required=True)
    p.add_argument("--renderer_ckpt",       required=True)
    # LoRA hyper-parameters
    p.add_argument("--lora_rank",     type=int,   default=32)
    p.add_argument("--lora_alpha",    type=float, default=64.0)
    p.add_argument("--lora_dropout",  type=float, default=0.05)
    # Training hyper-parameters
    p.add_argument("--batch_size",    type=int,   default=8)
    p.add_argument("--lr",            type=float, default=1e-4)
    p.add_argument("--max_steps",     type=int,   default=100_000)
    p.add_argument("--max_epochs",    type=int,   default=-1)
    p.add_argument("--save_freq",     type=int,   default=5_000)
    p.add_argument("--display_freq",  type=int,   default=2_000)
    p.add_argument("--log_every_n_steps", type=int, default=50)
    p.add_argument("--num_workers",   type=int,   default=4)
    p.add_argument("--val_split",     type=int,   default=100,
                   help="Number of samples held out for validation")
    p.add_argument("--resume_ckpt",   default=None,
                   help="Explicit emotion LoRA checkpoint to resume from")
    p.add_argument("--no_auto_resume", action="store_true")
    p.add_argument("--precision",      default="32-true",
                   help="pl.Trainer precision: 32-true | bf16-mixed | 16-mixed")
    # Per-epoch test video
    p.add_argument("--test_ref",      default=None)
    p.add_argument("--test_aud",      default=None)
    p.add_argument("--test_out_dir",  default=None)
    p.add_argument("--test_every_n_epochs", type=int, default=1)
    p.add_argument("--test_crop",     type=int, default=0,
                   help="0=do not crop (safe for robot faces); 1=auto-crop")
    p.add_argument("--test_cfg",      type=float, default=3.0)
    p.add_argument("--test_nfe",      type=int, default=10)
    p.add_argument("--test_seed",     type=int, default=42)
    p.add_argument("--test_emotion_id", type=int, default=1,
                   help="Emotion ID to demonstrate in the test video (default: 1 = happy)")
    # Emotion-specific
    p.add_argument("--emotion_filter", nargs="*", default=None,
                   help="Only train on specific emotions e.g. --emotion_filter happy sad")
    p.add_argument("--list_path",     default=None,
                   help="Explicit train list file (one stem per line)")
    p.add_argument("--val_list_path", default=None,
                   help="Explicit val list file")
    return p.parse_args(argv)


# -- Key-rename adapter (same as run_generator_training.py) ------------------

class _KeyRenameDataset(torch.utils.data.Dataset):
    """dataset.py returns 'gaze_now' but FM.py reads 'gaze'; fix that."""
    RENAME = (("gaze_now", "gaze"), ("pose_now", "pose"), ("cam_now", "cam"))

    def __init__(self, ds):
        self.ds = ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        b = self.ds[i]
        for k_from, k_to in self.RENAME:
            if k_from in b:
                b[k_to] = b[k_from]
        return b


# -- Lightning system ---------------------------------------------------------

def _build_system(opt, base_generator_ckpt: Path, style_lora_ckpt: Path,
                  lora_rank: int, lora_alpha: float, lora_dropout: float):
    """Build the LightningModule with frozen base + emotion LoRA."""
    import pytorch_lightning as pl
    from torch import nn, optim
    from generator.FM import FMGenerator
    from generator.train import EMA, append_dims, L1loss
    from generator.train_lora import apply_emotion_lora_to_model, _get_child

    class EmotionLoRASystem(pl.LightningModule):
        def __init__(self):
            super().__init__()
            self.model = FMGenerator(opt)
            self.opt   = opt
            self.loss_fn = L1loss()

            # 1. Load pretrained base generator weights (generator.ckpt)
            base_path = Path(base_generator_ckpt)
            assert base_path.exists(), f"Base generator checkpoint not found: {base_path}"
            b_ckpt = torch.load(str(base_path), map_location="cpu", weights_only=False)
            b_raw  = (b_ckpt.get("ema_state_dict")
                     or b_ckpt.get("state_dict", b_ckpt.get("model", b_ckpt)))
            if isinstance(b_raw, dict) and "model" in b_raw and isinstance(b_raw["model"], dict):
                b_raw = b_raw["model"]
            b_state = {
                k.replace("model.", "", 1) if k.startswith("model.") else k: v
                for k, v in b_raw.items()
            }
            missing, unexpected = self.model.load_state_dict(b_state, strict=False)
            matched = len(set(self.model.state_dict().keys()) - set(missing))
            print(f"[EmotionLoRA] Base generator loaded from {base_path.name} | "
                  f"{matched} keys matched (missing={len(missing)}, unexpected={len(unexpected)})", flush=True)
            if matched < 100:
                raise RuntimeError(
                    f"CRITICAL ERROR: Only {matched} keys matched from base checkpoint {base_path}!\n"
                    f"Ensure base_generator_ckpt points to the full generator.ckpt, NOT a LoRA checkpoint."
                )

            # 2. Optionally merge style LoRA (e.g. ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt)
            if style_lora_ckpt and Path(style_lora_ckpt).exists():
                s_path = Path(style_lora_ckpt)
                s_ckpt = torch.load(str(s_path), map_location="cpu", weights_only=False)
                s_raw  = (s_ckpt.get("ema_state_dict")
                         or s_ckpt.get("state_dict", s_ckpt.get("model", s_ckpt)))
                if isinstance(s_raw, dict) and "model" in s_raw and isinstance(s_raw["model"], dict):
                    s_raw = s_raw["model"]
                s_state = {
                    k.replace("model.", "", 1) if k.startswith("model.") else k: v
                    for k, v in s_raw.items()
                }
                lora_targets = set()
                for k in s_state.keys():
                    if ".lora_A.weight" in k:
                        lora_targets.add(k.replace(".lora_A.weight", ""))
                
                if lora_targets:
                    style_scale = float(getattr(opt, "style_lora_alpha", 128.0)) / max(1, int(getattr(opt, "style_lora_rank", 64)))
                    merged = 0
                    for target in lora_targets:
                        A = s_state.get(f"{target}.lora_A.weight")
                        B = s_state.get(f"{target}.lora_B.weight")
                        if A is not None and B is not None:
                            delta_W = style_scale * (B @ A)
                            mod = _get_child(self.model, target)
                            mod.weight.data += delta_W.to(mod.weight.device)
                            merged += 1
                    print(f"[EmotionLoRA] Merged {merged} style LoRA layers from {s_path.name} (scale={style_scale:.2f}) into base generator weights", flush=True)

            # 3. Apply emotion LoRA (freezes base, adds adapters, unfreezes emotion_conditioner)
            apply_emotion_lora_to_model(
                self.model,
                rank=lora_rank,
                alpha=lora_alpha,
                dropout=lora_dropout,
            )

            # 4. EMA on trainable params only
            self.ema = EMA(self.model, decay=0.9999)

        def on_train_batch_end(self, outputs, batch, batch_idx):
            self.ema.update()

        def on_validation_epoch_start(self):
            self.ema.apply_shadow()

        def on_validation_epoch_end(self):
            self.ema.restore()

        def on_save_checkpoint(self, checkpoint):
            checkpoint["ema_state_dict"] = self.ema.shadow

        def on_load_checkpoint(self, checkpoint):
            if "ema_state_dict" in checkpoint:
                self.ema.shadow = checkpoint["ema_state_dict"]
                print("[EmotionLoRA] EMA state restored from checkpoint", flush=True)

        def _shared_step(self, batch):
            m_now = batch["m_now"]
            noise  = torch.randn_like(m_now)
            times  = torch.rand(m_now.size(0), device=self.device)
            t      = append_dims(times, m_now.ndim - 1)
            noised = t * m_now + (1 - t) * noise
            gt_flow = m_now - noise
            batch["m_now"] = noised
            pred = self.model(batch, t=times)
            fm_loss  = self.loss_fn(pred, gt_flow)
            vel_loss = self.loss_fn(
                pred[:, 1:] - pred[:, :-1],
                gt_flow[:, 1:] - gt_flow[:, :-1],
            )
            return fm_loss + vel_loss, fm_loss

        def training_step(self, batch, batch_idx):
            loss, fm = self._shared_step(batch)
            self.log("train_loss", loss, prog_bar=True)
            self.log("fm_loss",    fm,   prog_bar=True)
            return loss

        def validation_step(self, batch, batch_idx):
            loss, fm = self._shared_step(batch)
            self.log("val_loss",    loss, prog_bar=True)
            self.log("val_fm_loss", fm,   prog_bar=True)

        def configure_optimizers(self):
            emo_params  = list(self.model.emotion_conditioner.parameters())
            lora_params = [p for n, p in self.model.named_parameters()
                           if p.requires_grad and "emotion_conditioner" not in n]
            optimizer = optim.AdamW([
                {"params": emo_params,  "lr": self.opt.lr * 2.0},
                {"params": lora_params, "lr": self.opt.lr},
            ], betas=(0.9, 0.999), weight_decay=1e-4)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=self.opt.iter, eta_min=1e-6
            )
            return {"optimizer": optimizer, "lr_scheduler": scheduler}

    return EmotionLoRASystem()


# -- Main ---------------------------------------------------------------------

def main(args):
    repo_root = Path(args.repo_root).resolve()
    gen_dir   = repo_root / "generator"
    for p in (str(repo_root), str(gen_dir)):
        if p not in sys.path:
            sys.path.insert(0, p)

    import pytorch_lightning as pl
    from pytorch_lightning.loggers import TensorBoardLogger, CSVLogger
    from generator.emotion_dataset import EmotionAudioMotionDataset, NUM_EMOTIONS

    # Build opt (reuse TrainOptions structure for all model hyper-params)
    import argparse as _ap
    from generator.train import TrainOptions
    parser = _ap.ArgumentParser()
    parser = TrainOptions().initialize(parser)
    opt = parser.parse_args([])
    opt.dataset_path      = args.dataset_path
    opt.batch_size        = args.batch_size
    opt.lr                = args.lr
    opt.iter              = args.max_steps
    opt.save_freq         = args.save_freq
    opt.display_freq      = args.display_freq
    opt.rank              = "cuda"
    opt.wav2vec_model_path = args.wav2vec_model_path
    opt.fix_noise_seed    = True
    opt.seed              = args.test_seed
    opt.skip_fm_audio_encoder = True   # dataset has precomputed .npy features

    exp_root = Path(args.exp_root).resolve()
    exp_dir  = exp_root / args.exp_name
    (exp_dir / "checkpoints").mkdir(parents=True, exist_ok=True)
    opt.exp_path = str(exp_root)
    opt.exp_name = args.exp_name

    # -- Checkpoint Resolution ------------------------------------------------
    base_generator_ckpt = getattr(args, "base_generator_ckpt", None)
    style_lora_ckpt     = getattr(args, "style_lora_ckpt", None)
    pretrained_ckpt     = getattr(args, "pretrained_ckpt", None)

    if not base_generator_ckpt:
        if pretrained_ckpt:
            p_path = Path(pretrained_ckpt)
            if p_path.exists():
                p_raw = torch.load(str(p_path), map_location="cpu", weights_only=False)
                p_state = p_raw.get("ema_state_dict") or p_raw.get("state_dict", p_raw)
                if isinstance(p_state, dict) and "model" in p_state and isinstance(p_state["model"], dict):
                    p_state = p_state["model"]
                has_lora_keys = any("lora_" in k for k in p_state.keys())
                has_base_keys = any("fmt.blocks.0.attn.qkv.weight" in k for k in p_state.keys())
                
                if has_lora_keys and not has_base_keys:
                    # It's a LoRA checkpoint!
                    style_lora_ckpt = p_path
                    for cand in [
                        repo_root / "checkpoints" / "generator.ckpt",
                        repo_root.parent / "checkpoints" / "generator.ckpt",
                        repo_root.parent.parent / "checkpoints" / "generator.ckpt",
                    ]:
                        if cand.exists():
                            base_generator_ckpt = cand
                            break
                    print(f"[Pretrained] Note: {p_path.name} is a LoRA checkpoint. Auto-paired with base generator: {base_generator_ckpt}", flush=True)
                else:
                    base_generator_ckpt = p_path

        if not base_generator_ckpt:
            for cand in [
                repo_root / "checkpoints" / "generator.ckpt",
                repo_root.parent / "checkpoints" / "generator.ckpt",
            ]:
                if cand.exists():
                    base_generator_ckpt = cand
                    break

    assert base_generator_ckpt and Path(base_generator_ckpt).exists(), \
        f"Base generator checkpoint (generator.ckpt) not found: {base_generator_ckpt}\n" \
        "Please provide --base_generator_ckpt pointing to generator.ckpt"

    print(f"[Pretrained] Base generator: {base_generator_ckpt}", flush=True)
    if style_lora_ckpt:
        print(f"[Pretrained] Style LoRA to merge: {style_lora_ckpt}", flush=True)

    opt.style_lora_rank  = getattr(args, "style_lora_rank", 64)
    opt.style_lora_alpha = getattr(args, "style_lora_alpha", 128.0)

    # -- Datasets -------------------------------------------------------------
    all_motion = sorted(Path(args.dataset_path).glob("motion/*.pt"))
    n_total    = len(all_motion)
    val_count  = args.val_split if n_total > args.val_split + 10 else max(1, n_total // 10)
    print(f"[Data] {n_total} clips -> train={n_total - val_count} val={val_count}", flush=True)

    ds_train = _KeyRenameDataset(EmotionAudioMotionDataset(
        opt=opt, start=0, end=-val_count, split="train",
        list_path=args.list_path,
        emotion_filter=args.emotion_filter,
    ))
    ds_val = _KeyRenameDataset(EmotionAudioMotionDataset(
        opt=opt, start=-val_count, end=None, split="val",
        list_path=args.val_list_path,
        emotion_filter=args.emotion_filter,
    ))

    dl_train = torch.utils.data.DataLoader(
        ds_train, batch_size=opt.batch_size, shuffle=True,
        num_workers=args.num_workers,
        persistent_workers=args.num_workers > 0,
    )
    dl_val = torch.utils.data.DataLoader(
        ds_val, batch_size=min(8, opt.batch_size), shuffle=False,
        num_workers=0,
    )

    # -- System ---------------------------------------------------------------
    system = _build_system(
        opt=opt,
        base_generator_ckpt=base_generator_ckpt,
        style_lora_ckpt=style_lora_ckpt,
        lora_rank=args.lora_rank,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
    )

    # -- Resume logic ---------------------------------------------------------
    ckpt_path = None
    if args.resume_ckpt and Path(args.resume_ckpt).exists():
        ckpt_path = args.resume_ckpt
    elif not args.no_auto_resume:
        last = exp_dir / "checkpoints" / "last.ckpt"
        if last.exists():
            ckpt_path = str(last)
    if ckpt_path:
        print(f"[Resume] {ckpt_path}", flush=True)

    # -- Callbacks / loggers --------------------------------------------------
    tb_logger  = TensorBoardLogger(save_dir=str(exp_root), name=args.exp_name)
    csv_logger = CSVLogger(save_dir=str(exp_root), name=args.exp_name)

    # Text log callback (reuse from imtalker_train_utils)
    text_log = None
    try:
        try:
            from generator.imtalker_train_utils import TextLogCallback
        except ImportError:
            for sp in (str(repo_root / "preprocessing"), str(repo_root / "generator")):
                if sp not in sys.path:
                    sys.path.insert(0, sp)
            from imtalker_train_utils import TextLogCallback
        text_log = TextLogCallback(exp_dir / "train_log.txt", every_n_steps=args.log_every_n_steps)
    except Exception as e:
        print(f"[warn] TextLogCallback not initialized: {e}", flush=True)
        text_log = None

    test_video_cb = None
    if args.test_ref and args.test_aud and Path(args.test_ref).exists() and Path(args.test_aud).exists():
        try:
            try:
                from generator.imtalker_train_utils import EpochTestVideoCallback
            except ImportError:
                from imtalker_train_utils import EpochTestVideoCallback
            test_video_cb = EpochTestVideoCallback(
                ref_path=args.test_ref,
                aud_path=args.test_aud,
                out_dir=args.test_out_dir or str(exp_dir / "test_videos"),
                renderer_ckpt=args.renderer_ckpt,
                wav2vec_model_path=args.wav2vec_model_path,
                every_n_epochs=args.test_every_n_epochs,
                crop=bool(args.test_crop),
                a_cfg_scale=args.test_cfg,
                nfe=args.test_nfe,
                seed=args.test_seed,
                test_emotion_id=getattr(args, "test_emotion_id", 1),
            )
            print(f"[Callback] EpochTestVideoCallback enabled | every {args.test_every_n_epochs} epochs", flush=True)
        except Exception as e:
            print(f"[warn] EpochTestVideoCallback not initialized: {e}", flush=True)

    checkpoint_callback = pl.callbacks.ModelCheckpoint(
        dirpath=str(exp_dir / "checkpoints"),
        filename="{step:06d}",
        every_n_train_steps=opt.save_freq,
        save_top_k=-1,
        save_last=True,
    )
    lr_monitor = pl.callbacks.LearningRateMonitor(logging_interval="step")
    callbacks  = [checkpoint_callback, lr_monitor]
    if text_log:
        callbacks.append(text_log)
    if test_video_cb:
        callbacks.append(test_video_cb)

    # -- val_check_interval safety ---------------------------------------------
    n_gpu = max(1, torch.cuda.device_count())
    steps_per_epoch = max(1, math.ceil(len(ds_train) / (opt.batch_size * n_gpu)))
    display_freq    = min(opt.display_freq, steps_per_epoch)

    # -- Trainer --------------------------------------------------------------
    strategy = "ddp_find_unused_parameters_true" if torch.cuda.device_count() > 1 else "auto"
    print(f"[Trainer] gpus={n_gpu} strategy={strategy} max_steps={opt.iter} "
          f"precision={args.precision} val_check_interval={display_freq}", flush=True)

    trainer = pl.Trainer(
        accelerator="gpu",
        devices=-1,
        strategy=strategy,
        precision=args.precision,
        max_steps=opt.iter,
        max_epochs=args.max_epochs,
        val_check_interval=display_freq,
        check_val_every_n_epoch=None,
        logger=[tb_logger, csv_logger],
        callbacks=callbacks,
        enable_progress_bar=True,
    )

    trainer.fit(system, dl_train, dl_val, ckpt_path=ckpt_path)
    print("[Done] emotion LoRA training finished", flush=True)


if __name__ == "__main__":
    main(parse_args())
'''

_dst = REPO_ROOT / "generator/run_emotion_lora_training.py"
_dst.write_text(_SRC.lstrip("\n"), encoding="utf-8")
compile(_SRC, str(_dst), "exec")
print(f"[write] {_dst} ({len(_SRC.splitlines())} lines)")


In [ ]:
# ============================================================================
# CREATE generator/test_emotions.py (identical to the old pipeline file)
# ============================================================================
from pathlib import Path

_SRC = r'''
"""
Render talking videos across all 6 emotions for a given emotion LoRA checkpoint.

Usage:
    python generator/test_emotions.py \
        --checkpoint /path/to/emotion_exps/emotion_lora_v1/checkpoints/last.ckpt \
        --ref_img assets/3robert.jpeg \
        --audio assets/audio_1.wav \
        --out_dir test_renders \
        --cfg 3.0
"""
from __future__ import annotations

import argparse
import math
import os
import sys
import tempfile
import subprocess
from pathlib import Path

import torch
import torch.nn.functional as F

# Fix weights_only in torch >= 2.6
_TORCH_LOAD = torch.load
def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _TORCH_LOAD(*args, **kwargs)
torch.load = _torch_load_compat


def parse_args():
    p = argparse.ArgumentParser(description="Render test videos for all 6 emotions")
    p.add_argument("--checkpoint", required=True, help="Trained emotion LoRA checkpoint (.ckpt)")
    p.add_argument("--base_generator_ckpt", default=None, help="Base generator.ckpt (auto-detected if None)")
    p.add_argument("--style_lora_ckpt", default=None, help="Optional live winner style LoRA")
    p.add_argument("--ref_img", default="assets/3robert.jpeg")
    p.add_argument("--audio", default="assets/audio_1.wav")
    p.add_argument("--out_dir", default="test_emotion_renders")
    p.add_argument("--renderer_ckpt", default=None)
    p.add_argument("--wav2vec_model_path", default=None)
    p.add_argument("--lora_rank", type=int, default=32)
    p.add_argument("--lora_alpha", type=float, default=64.0)
    p.add_argument("--cfg", type=float, default=3.0)
    p.add_argument("--nfe", type=int, default=10)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--emotions", nargs="*", default=None,
                   help="Specific emotions to test, e.g. --emotions happy sad angry")
    return p.parse_args()


def save_video(vid_tensor: torch.Tensor, video_path: Path, audio_path: str):
    import torchvision
    video_path = Path(video_path)
    video_path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        temp_filename = tmp.name

    vid = vid_tensor.permute(0, 2, 3, 1).detach().clamp(0, 1).cpu()
    torchvision.io.write_video(temp_filename, (vid * 255).to(torch.uint8), fps=25)

    cmd = (
        f'ffmpeg -y -v warning -i "{temp_filename}" -i "{audio_path}" '
        f'-c:v copy -c:a aac -shortest "{video_path}"'
    )
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if os.path.exists(temp_filename):
        os.remove(temp_filename)
    if res.returncode != 0:
        import shutil
        shutil.copy2(temp_filename, video_path)


def main():
    args = parse_args()
    repo_root = Path(__file__).resolve().parent.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    if str(repo_root / "generator") not in sys.path:
        sys.path.insert(0, str(repo_root / "generator"))

    from generator.options.base_options import BaseOptions
    from generator.FM import FMGenerator, EMOTION_MAP, EMOTION_NAMES
    from generator.train_lora import apply_emotion_lora_to_model, _get_child
    from generator.generate import DataProcessor, InferenceOptions
    from renderer.models import IMTRenderer
    from generator.wav2vec2 import Wav2VecModel

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Checkpoint paths
    ckpt_dir = repo_root / "checkpoints"
    renderer_ckpt = Path(args.renderer_ckpt or (ckpt_dir / "renderer.ckpt"))
    w2v_path = Path(args.wav2vec_model_path or (ckpt_dir / "wav2vec2-base-960h"))
    base_gen = Path(args.base_generator_ckpt or (ckpt_dir / "generator.ckpt"))

    assert renderer_ckpt.exists(), f"renderer missing: {renderer_ckpt}"
    assert w2v_path.exists(), f"wav2vec missing: {w2v_path}"
    assert base_gen.exists(), f"base generator missing: {base_gen}"

    # Build model
    parser = argparse.ArgumentParser()
    parser = BaseOptions().initialize(parser)
    parser.add_argument("--skip_fm_audio_encoder", action="store_true")
    opt = parser.parse_args([])
    opt.rank = device
    opt.wav2vec_model_path = str(w2v_path)
    opt.skip_fm_audio_encoder = True
    opt.fix_noise_seed = True
    opt.seed = args.seed

    model = FMGenerator(opt).to(device)

    # 1. Load base weights
    b_ckpt = torch.load(str(base_gen), map_location="cpu")
    b_raw  = b_ckpt.get("ema_state_dict") or b_ckpt.get("state_dict", b_ckpt)
    if "model" in b_raw and isinstance(b_raw["model"], dict): b_raw = b_raw["model"]
    b_state = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in b_raw.items()}
    model.load_state_dict(b_state, strict=False)

    # 2. Merge style LoRA if present
    if args.style_lora_ckpt and Path(args.style_lora_ckpt).exists():
        s_path = Path(args.style_lora_ckpt)
        s_ckpt = torch.load(str(s_path), map_location="cpu")
        s_raw  = s_ckpt.get("ema_state_dict") or s_ckpt.get("state_dict", s_ckpt)
        if "model" in s_raw and isinstance(s_raw["model"], dict): s_raw = s_raw["model"]
        s_state = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in s_raw.items()}
        lora_targets = {k.replace(".lora_A.weight", "") for k in s_state if ".lora_A.weight" in k}
        style_scale = 128.0 / 64.0
        for target in lora_targets:
            A = s_state.get(f"{target}.lora_A.weight")
            B = s_state.get(f"{target}.lora_B.weight")
            if A is not None and B is not None:
                mod = _get_child(model, target)
                mod.weight.data += (style_scale * (B @ A)).to(mod.weight.device)
        print(f"Merged {len(lora_targets)} style LoRA layers")

    # 3. Apply emotion LoRA
    apply_emotion_lora_to_model(model, rank=args.lora_rank, alpha=args.lora_alpha, dropout=0.0)

    # 4. Load trained emotion checkpoint
    e_ckpt = torch.load(str(args.checkpoint), map_location="cpu")
    e_raw  = e_ckpt.get("ema_state_dict") or e_ckpt.get("state_dict", e_ckpt)
    if "model" in e_raw and isinstance(e_raw["model"], dict): e_raw = e_raw["model"]
    e_state = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in e_raw.items()}
    missing, unexpected = model.load_state_dict(e_state, strict=False)
    print(f"Loaded emotion checkpoint: {Path(args.checkpoint).name} (missing={len(missing)}, unexpected={len(unexpected)})")
    model.eval()

    # Load Renderer & Wav2Vec
    p_inf = argparse.ArgumentParser()
    p_inf = InferenceOptions().initialize(p_inf)
    opt_inf = p_inf.parse_args([])
    opt_inf.rank = device
    opt_inf.renderer_path = str(renderer_ckpt)
    opt_inf.wav2vec_model_path = str(w2v_path)

    renderer = IMTRenderer(opt_inf).to(device).eval()
    r_ckpt = torch.load(str(renderer_ckpt), map_location="cpu")
    r_state = {k.replace("gen.", ""): v for k, v in r_ckpt["state_dict"].items() if k.startswith("gen.")}
    renderer.load_state_dict(r_state, strict=False)

    wav2vec = Wav2VecModel.from_pretrained(str(w2v_path), local_files_only=True).to(device).eval()
    data_processor = DataProcessor(opt_inf)

    # Preprocess reference image and audio
    ref_path = Path(args.ref_img)
    if not ref_path.is_absolute():
        ref_path = repo_root / ref_path
    aud_path = Path(args.audio)
    if not aud_path.is_absolute():
        aud_path = repo_root / aud_path

    print(f"Reference image: {ref_path}")
    print(f"Audio file:      {aud_path}")

    data = data_processor.preprocess(str(ref_path), str(aud_path), crop=False)
    s = data["s"].to(device)
    f_r, g_r = renderer.dense_feature_encoder(s)
    t_r = renderer.latent_token_encoder(s)
    data["ref_x"] = t_r
    data["pose"] = None
    data["cam"] = None
    data["gaze"] = None

    # Wav2Vec features
    raw_audio = data["a"].squeeze(0).to(device)
    T_target = math.ceil(raw_audio.shape[0] * 25 / 16000)
    target_samples = int(T_target * 16000 / 25)
    if raw_audio.shape[0] < target_samples:
        diff = target_samples - raw_audio.shape[0]
        raw_audio = F.pad(raw_audio.unsqueeze(0), (0, diff), mode="replicate").squeeze(0)
    else:
        raw_audio = raw_audio[:target_samples]
    with torch.no_grad():
        out = wav2vec(raw_audio.unsqueeze(0), seq_len=T_target, output_hidden_states=False)
        feat_25 = out.last_hidden_state
    data["a_feat"] = feat_25

    # Determine which emotions to render
    if args.emotions:
        test_eids = []
        for e in args.emotions:
            e_clean = e.strip().lower()
            if e_clean.isdigit():
                test_eids.append(int(e_clean))
            elif e_clean in EMOTION_MAP:
                test_eids.append(EMOTION_MAP[e_clean])
    else:
        test_eids = [0, 1, 2, 3, 4, 5]  # all 6 emotions

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nGenerating videos for emotions: {[EMOTION_NAMES.get(e, e) for e in test_eids]} (cfg={args.cfg}) ...")

    for eid in test_eids:
        emo_name = EMOTION_NAMES.get(eid, f"emotion_{eid}")
        print(f"  Rendering [{eid}] {emo_name} ...", flush=True)
        data["emotion_id"] = eid

        with torch.no_grad():
            sample = model.sample(data, a_cfg_scale=args.cfg, nfe=args.nfe, seed=args.seed)

            # Decode
            T = sample.shape[1]
            ta_r = renderer.adapt(t_r, g_r)
            m_r = renderer.latent_token_decoder(ta_r)
            d_hat = []
            for t in range(T):
                ta_c = renderer.adapt(sample[:, t, ...], g_r)
                m_c = renderer.latent_token_decoder(ta_c)
                d_i = renderer.decode(m_c, m_r, f_r)
                d_hat.append(d_i)

            vid_tensor = torch.cat(d_hat, dim=0)
            out_file = out_dir / f"{eid}_{emo_name}.mp4"
            save_video(vid_tensor, out_file, str(aud_path))
            print(f"    [OK] Saved -> {out_file}")

    print(f"\nAll renders saved in: {out_dir.resolve()}")


if __name__ == "__main__":
    main()
'''

_dst = REPO_ROOT / "generator/test_emotions.py"
_dst.write_text(_SRC.lstrip("\n"), encoding="utf-8")
compile(_SRC, str(_dst), "exec")
print(f"[write] {_dst} ({len(_SRC.splitlines())} lines)")


In [ ]:
# ============================================================================
# Verify the emotion adapter code is importable
# ============================================================================
import importlib.util
import inspect
import sys
from pathlib import Path

for _p in (str(REPO_ROOT), str(REPO_ROOT / "generator")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from generator.FM import FMGenerator, EmotionConditioner, EMOTION_MAP, EMOTION_NAMES, NUM_EMOTIONS
print("[OK] generator/FM.py: FMGenerator + EmotionConditioner + EMOTION_MAP/NAMES")

from generator.FMT import FlowMatchingTransformer
assert "emotion_bias_now" in inspect.signature(FlowMatchingTransformer.forward).parameters
for _k in ("emotion_bias_now", "null_emotion_bias_now"):
    assert _k in inspect.signature(FlowMatchingTransformer.forward_with_cfg).parameters, _k
print("[OK] generator/FMT.py: emotion bias in forward()/forward_with_cfg()")

from generator.train_lora import apply_emotion_lora_to_model, apply_lora_to_model, LoRALinear
print("[OK] generator/train_lora.py: apply_emotion_lora_to_model()")

from generator.emotion_dataset import EmotionAudioMotionDataset, parse_emotion_from_stem
print("[OK] generator/emotion_dataset.py: EmotionAudioMotionDataset")

import generator.imtalker_train_utils as _itu
assert hasattr(_itu, "EpochTestVideoCallback") and hasattr(_itu, "TextLogCallback")
print("[OK] generator/imtalker_train_utils.py: TextLogCallback + EpochTestVideoCallback")

_spec = importlib.util.spec_from_file_location(
    "run_emotion_lora_training", str(REPO_ROOT / "generator" / "run_emotion_lora_training.py"))
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
assert hasattr(_mod, "main") and hasattr(_mod, "parse_args")
print("[OK] generator/run_emotion_lora_training.py: main()/parse_args()")

_spec2 = importlib.util.spec_from_file_location(
    "test_emotions", str(REPO_ROOT / "generator" / "test_emotions.py"))
_mod2 = importlib.util.module_from_spec(_spec2)
_spec2.loader.exec_module(_mod2)
assert hasattr(_mod2, "main")
print("[OK] generator/test_emotions.py: main()")

print("\nAll emotion adapter code verified.")


## Download pretrained models and test files

Downloads the base IMTalker checkpoints from Hugging Face (`cbsjtu01/IMTalker`) --
exactly the same files the new pipeline's `download_live_assets.sh` script uses:

- `checkpoints/generator.ckpt` -- the full ~38.8M-param pretrained FM generator (base)
- `checkpoints/renderer.ckpt` -- the face renderer
- `checkpoints/wav2vec2-base-960h/` -- the audio feature extractor

It also resolves the test assets `3robert.jpeg` + `audio_3.wav` (used for the
every-10-epoch test renders): files placed next to this notebook take priority;
otherwise the identical copies are downloaded from the old pipeline repository.


In [ ]:
# ============================================================================
# Download all required pretrained models & checkpoints (autonomous)
# ============================================================================
import shutil
import urllib.request
from pathlib import Path
from huggingface_hub import hf_hub_download

CKPT_DIR.mkdir(parents=True, exist_ok=True)
(REPO_ROOT / "assets").mkdir(parents=True, exist_ok=True)

# --- 1. Base IMTalker files (config.yaml, renderer.ckpt, generator.ckpt) -----
IMTALKER_FILES = ["config.yaml", "renderer.ckpt", "generator.ckpt"]
for fname in IMTALKER_FILES:
    dest = CKPT_DIR / fname
    if not dest.exists():
        print(f"[download] {fname} from cbsjtu01/IMTalker ...")
        hf_hub_download(repo_id="cbsjtu01/IMTalker", filename=fname, local_dir=str(CKPT_DIR))
        print(f"  [OK] {fname} -> {dest}")
    else:
        print(f"[skip] {fname} already present: {dest}")

# --- 2. wav2vec2-base-960h ---------------------------------------------------
WAV2VEC_DIR.mkdir(parents=True, exist_ok=True)
W2V_FILES = [
    "wav2vec2-base-960h/config.json",
    "wav2vec2-base-960h/pytorch_model.bin",
    "wav2vec2-base-960h/preprocessor_config.json",
    "wav2vec2-base-960h/feature_extractor_config.json",
]
for rel_path in W2V_FILES:
    dest = CKPT_DIR / rel_path
    if not dest.exists():
        print(f"[download] {rel_path} from cbsjtu01/IMTalker ...")
        hf_hub_download(repo_id="cbsjtu01/IMTalker", filename=rel_path, local_dir=str(CKPT_DIR))
        print(f"  [OK] {rel_path}")
    else:
        print(f"[skip] {rel_path} already present")

# --- 3. Optional style LoRA (old pipeline's live-winner blink LoRA) ----------
if STYLE_LORA_CKPT and not Path(STYLE_LORA_CKPT).exists():
    winner_rel = "live_winner/lora/ditto_blink_lora_withaudio_r64_096_continue_2h_last.ckpt"
    print("[download] style LoRA from niloy629/hdtf_preprocess ...")
    try:
        download_root = PROJECT_ROOT / "checkpoints"
        download_root.mkdir(parents=True, exist_ok=True)
        hf_hub_download(
            repo_id="niloy629/hdtf_preprocess",
            repo_type="dataset",
            filename=winner_rel,
            local_dir=str(download_root),
        )
    except Exception as e:
        print(f"[warn] style LoRA download failed: {e}")
        print("[info] training will use the plain base generator.ckpt")
        STYLE_LORA_CKPT = None

# --- 4. Test files (3robert.jpeg + audio_3.wav) ------------------------------
# Identical copies exist in the old pipeline repo (byte-identical, verified).
TEST_FILE_URLS = {
    "3robert.jpeg": "https://raw.githubusercontent.com/MoshiHead/n_IMT_s_Perso_v2_2_aug_compress_log_Emotion_anti_v1/main/IMTalker/assets/3robert.jpeg",
    "audio_3.wav": "https://raw.githubusercontent.com/MoshiHead/n_IMT_s_Perso_v2_2_aug_compress_log_Emotion_anti_v1/main/IMTalker/assets/audio_3.wav",
}

def _resolve_test_file(filename, configured):
    candidates = []
    if configured:
        candidates.append(Path(configured))
    candidates += [
        Path.cwd() / filename,                    # next to this notebook
        Path("/workspace") / filename,
        Path(REPO_DIR) / filename,
        REPO_ROOT / "assets" / filename,
    ]
    for c in candidates:
        if c.exists() and c.stat().st_size > 0:
            return c
    url = TEST_FILE_URLS[filename]
    dest = REPO_ROOT / "assets" / filename
    print(f"[download] {filename} from the old pipeline repo (identical file) ...")
    dest.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, dest)
    return dest

TEST_REF_IMAGE = _resolve_test_file("3robert.jpeg", TEST_REF_IMAGE)
TEST_AUDIO = _resolve_test_file("audio_3.wav", TEST_AUDIO)

# keep a copy inside the repo assets so the training scripts find it
for f in (TEST_REF_IMAGE, TEST_AUDIO):
    dst = REPO_ROOT / "assets" / f.name
    if f.resolve() != dst.resolve():
        shutil.copy2(str(f), str(dst))
        print(f"[copy] {f} -> {dst}")
    print(f"[test] {f.name}: {f} ({f.stat().st_size / 1e6:.2f} MB)")

# --- 5. Summary ----------------------------------------------------------------
print("\nModel checkpoints ready for training:")
print(f"  renderer.ckpt:        {RENDERER_CKPT} (exists={RENDERER_CKPT.exists()})")
print(f"  wav2vec2-base-960h:   {WAV2VEC_DIR} (exists={WAV2VEC_DIR.exists()})")
print(f"  base generator.ckpt:  {BASE_GENERATOR_CKPT} (exists={BASE_GENERATOR_CKPT.exists()})")
has_style = bool(STYLE_LORA_CKPT and Path(STYLE_LORA_CKPT).exists())
print(f"  style LoRA to merge:  {STYLE_LORA_CKPT} (exists={has_style})")
print(f"  test image:           {TEST_REF_IMAGE}")
print(f"  test audio:           {TEST_AUDIO}")

assert RENDERER_CKPT.exists(), f"renderer.ckpt missing at {RENDERER_CKPT}"
assert WAV2VEC_DIR.exists() and any(WAV2VEC_DIR.iterdir()), f"wav2vec2-base-960h missing at {WAV2VEC_DIR}"
assert BASE_GENERATOR_CKPT.exists(), f"Base generator.ckpt missing at {BASE_GENERATOR_CKPT}"
print("\n[OK] All required model checkpoints verified successfully!")


## Helium feature extraction -- why it is (not) needed here

The new pipeline audio path for the avatar at live time is:

```
PersonaPlex 7B "Helium" hidden states (4096-d @ 12.5 Hz)
    -> UniTalk last-layer adapter (weights/adapter_last.pt)
    -> 768-d Wav2Vec2-space features @ 25 Hz
    -> FMGenerator (created with skip_fm_audio_encoder=True, fed via data["a_feat"])
```

So the FMGenerator audio contract is **wav2vec2 `last_hidden_state` space** -- and that
is exactly what the emotion dataset stores in `audio/*.npy` (the same preprocessed
format the old pipeline used; the new `generator/dataset.py` reads it unchanged).

Consequences for emotion LoRA training:

* **No Helium pass is required for training.** The emotion LoRA finetunes the
  generator in wav2vec2 feature space -- identical to how the base generator was
  trained, and identical to how the old pipeline emotion LoRA was trained and then
  successfully deployed on the Helium -> UniTalk-adapter live path.
* At live time the UniTalk adapter keeps producing those same features from Helium,
  so the trained emotion adapter drops into the same generator without any change.

The cell below therefore implements the **feature extraction the new architecture
actually requires for the dataset**: computing wav2vec2 `audio/*.npy` features from
raw 16 kHz `.wav` files with the repository's own `Wav2VecModel` (25 fps, 768-d,
`last_hidden_state` -- the exact contract above). It is idempotent and only needed
if your dataset's `audio/` folder is missing `.npy` files.


In [ ]:
# ============================================================================
# OPTIONAL -- wav2vec2 feature extraction for the emotion dataset
# ----------------------------------------------------------------------------
# Only run this if dataset/audio/*.npy are missing for some clips (e.g. you
# added new raw .wav files). Existing .npy files are never overwritten.
# Place raw .wav files in {EMOTION_DATASET_DIR}/wav/{stem}.wav or
# {EMOTION_DATASET_DIR}/audio/{stem}.wav; features are written to
# {EMOTION_DATASET_DIR}/audio/{stem}.npy
# ============================================================================
import math
import sys
from pathlib import Path

RUN_FEATURE_EXTRACTION = False   # <- set True to extract missing features

if RUN_FEATURE_EXTRACTION:
    for _p in (str(REPO_ROOT), str(REPO_ROOT / "generator")):
        if _p not in sys.path:
            sys.path.insert(0, _p)

    import librosa
    import numpy as np
    import torch
    import torch.nn.functional as F
    from generator.wav2vec2 import Wav2VecModel

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    wav2vec = Wav2VecModel.from_pretrained(str(WAV2VEC_DIR), local_files_only=True).to(device).eval()
    wav2vec.feature_extractor._freeze_parameters()

    audio_dir = EMOTION_DATASET_DIR / "audio"
    audio_dir.mkdir(parents=True, exist_ok=True)

    wavs = []
    for d in (EMOTION_DATASET_DIR / "wav", audio_dir):
        if d.exists():
            wavs.extend(sorted(d.glob("*.wav")))
    print(f"found {len(wavs)} .wav files")

    done = skipped = 0
    for wav_path in wavs:
        out_npy = audio_dir / f"{wav_path.stem}.npy"
        if out_npy.exists():
            skipped += 1
            continue
        try:
            speech, sr = librosa.load(str(wav_path), sr=16000)
            raw = torch.from_numpy(speech).float().to(device)
            # 25 fps feature grid, same padding as the live/test path
            T = int(math.ceil(raw.shape[0] * 25 / 16000))
            target_samples = int(T * 16000 / 25)
            if raw.shape[0] < target_samples:
                diff = target_samples - raw.shape[0]
                raw = F.pad(raw.unsqueeze(0), (0, diff), mode="replicate").squeeze(0)
            else:
                raw = raw[:target_samples]
            with torch.no_grad():
                out = wav2vec(raw.unsqueeze(0), seq_len=T, output_hidden_states=False)
                feat = out.last_hidden_state[0].float().cpu().numpy()   # [T, 768]
            np.save(str(out_npy), feat)
            done += 1
            print(f"  [OK] {wav_path.stem}: {feat.shape}")
        except Exception as e:
            print(f"  [fail] {wav_path.stem}: {e}")

    print(f"\nextracted={done} skipped(existing)={skipped}")
else:
    print("RUN_FEATURE_EXTRACTION = False -- skipping (dataset audio/*.npy already precomputed)")


In [ ]:
# ============================================================================
# PRE-FLIGHT CHECKS: dataset, checkpoints, emotion code, test files
# ============================================================================
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import torch

for _p in (str(REPO_ROOT), str(REPO_ROOT / "generator")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# 1) emotion conditioning code importable
from generator.FM import FMGenerator, EmotionConditioner
print("[OK] EmotionConditioner found in generator/FM.py")

from generator.emotion_dataset import (
    EmotionAudioMotionDataset, EMOTION_MAP, EMOTION_NAMES, parse_emotion_from_stem,
)
print("[OK] generator/emotion_dataset.py found")

training_script = REPO_ROOT / "generator" / "run_emotion_lora_training.py"
assert training_script.exists(), f"Missing {training_script}"
print(f"[OK] {training_script.name} found")

from generator.train_lora import apply_emotion_lora_to_model
print("[OK] apply_emotion_lora_to_model found in generator/train_lora.py")

# 2) dataset directory structure
for sub in ("motion", "audio", "smirk", "gaze"):
    d = EMOTION_DATASET_DIR / sub
    assert d.exists(), f"Missing {d} -- prepare your emotion dataset in {EMOTION_DATASET_DIR}"
counts = {s: len(list((EMOTION_DATASET_DIR / s).glob("*.*"))) for s in ("motion", "audio", "smirk", "gaze")}
print(f"\nDataset file counts: {counts}")
assert counts["motion"] >= max(10, VAL_SPLIT + 1), f"Need at least {VAL_SPLIT + 10} clips, found {counts['motion']}"

# 3) emotion label distribution
motion_files = sorted((EMOTION_DATASET_DIR / "motion").glob("*.pt"))
emotion_distribution = Counter(parse_emotion_from_stem(mp.stem) for mp in motion_files)
print("\nEmotion distribution in dataset:")
for eid in range(6):
    cnt = emotion_distribution.get(eid, 0)
    bar = "#" * (cnt // 50) if cnt > 0 else "(missing)"
    print(f"  [{eid}] {EMOTION_NAMES[eid]:10s}: {cnt:5d}  {bar}")
assert sum(emotion_distribution.values()) > 0, "No valid emotion labels found in dataset"

# 4) verify the base generator has the full weights (not a LoRA-only checkpoint)
base_raw = torch.load(str(BASE_GENERATOR_CKPT), map_location="cpu", weights_only=False)
base_dict = base_raw.get("ema_state_dict") or base_raw.get("state_dict", base_raw)
if isinstance(base_dict, dict) and "model" in base_dict and isinstance(base_dict["model"], dict):
    base_dict = base_dict["model"]
base_keys_count = len([k for k in base_dict.keys() if "fmt.blocks" in k])
assert base_keys_count > 50, (
    f"BASE_GENERATOR_CKPT ({BASE_GENERATOR_CKPT}) only has {base_keys_count} transformer keys! "
    "Must be the full generator.ckpt.")
print(f"\n[OK] Base generator verified: {len(base_dict)} total keys ({base_keys_count} transformer blocks)")
del base_raw, base_dict

# 5) test files
assert Path(TEST_REF_IMAGE).exists(), f"Test image missing: {TEST_REF_IMAGE}"
assert Path(TEST_AUDIO).exists(), f"Test audio missing: {TEST_AUDIO}"
print(f"[OK] test image: {TEST_REF_IMAGE}")
print(f"[OK] test audio: {TEST_AUDIO}")

# 6) a sample audio feature has the right shape (768-d wav2vec2 space)
sample_stem = motion_files[0].stem
audio_npy = EMOTION_DATASET_DIR / "audio" / f"{sample_stem}.npy"
assert audio_npy.exists(), f"Missing {audio_npy} -- run the feature extraction cell above"
feat = np.load(str(audio_npy), mmap_mode="r")
print(f"\n[OK] sample {sample_stem}: audio feature shape = {feat.shape} (expect [T, 768])")
assert feat.ndim == 2 and feat.shape[1] == 768, (
    f"audio features must be [T, 768] wav2vec2 last_hidden_state, got {feat.shape}")

print("\nAll pre-flight checks passed -- ready to train.")


## Launch training

**Option A (recommended)** runs `run_emotion_lora_training.py` as a background
process -- required for multi-GPU DDP and robust against kernel restarts. Use the
MONITOR cell to follow progress.

**Option B** runs the same script in-notebook (single GPU only; blocks the kernel).

The launcher:
1. loads the base generator checkpoint (full pretrained weights),
2. optionally merges a style LoRA checkpoint (e.g. the old live-winner blink LoRA),
3. applies the emotion LoRA setup (freeze all -> unfreeze emotion_conditioner ->
   LoRA on `fmt.c_embedder`, all `fmt.blocks.*.adaLN_modulation.1`, `fmt.decoder.*`),
4. trains with EMA + AdamW (2x LR on the conditioner) + cosine schedule,
5. runs validation every `display_freq` steps,
6. renders a test video with `3robert.jpeg` + `audio_3.wav` every **10 epochs**
   using the latest (EMA) weights -- training is never interrupted.


In [ ]:
# OPTION A -- Launch emotion LoRA training as a background process.
# Required for multi-GPU; survives kernel restarts.
import os
import subprocess
import sys
from pathlib import Path

training_script = REPO_ROOT / "generator" / "run_emotion_lora_training.py"
console_log = EXP_DIR / "train_console.log"
EXP_DIR.mkdir(parents=True, exist_ok=True)

cmd = [sys.executable, str(training_script),
    "--repo_root",            str(REPO_ROOT),
    "--dataset_path",         str(EMOTION_DATASET_DIR),
    "--base_generator_ckpt",  str(BASE_GENERATOR_CKPT),
    "--exp_root",             str(EXP_ROOT),
    "--exp_name",             EXP_NAME,
    "--wav2vec_model_path",   str(WAV2VEC_DIR),
    "--renderer_ckpt",        str(RENDERER_CKPT),
    "--lora_rank",            str(LORA_RANK),
    "--lora_alpha",           str(LORA_ALPHA),
    "--lora_dropout",         str(LORA_DROPOUT),
    "--batch_size",           str(BATCH_SIZE),
    "--lr",                   str(LR),
    "--max_steps",            str(MAX_STEPS),
    "--max_epochs",           str(MAX_EPOCHS),
    "--save_freq",            str(SAVE_FREQ),
    "--display_freq",         str(DISPLAY_FREQ),
    "--log_every_n_steps",    str(LOG_EVERY),
    "--num_workers",          str(NUM_WORKERS),
    "--val_split",            str(VAL_SPLIT),
    "--precision",            str(MIXED_PRECISION),
    "--test_every_n_epochs",  str(TEST_EVERY_N),
    "--test_crop",            "1" if TEST_CROP else "0",
    "--test_emotion_id",      str(TEST_EMOTION_ID),
    "--test_cfg",             str(TEST_CFG_SCALE),
    "--test_nfe",             str(TEST_NFE),
    "--test_seed",            str(TEST_SEED),
]
if STYLE_LORA_CKPT and Path(STYLE_LORA_CKPT).exists():
    cmd += [
        "--style_lora_ckpt",  str(STYLE_LORA_CKPT),
        "--style_lora_rank",  str(STYLE_LORA_RANK),
        "--style_lora_alpha", str(STYLE_LORA_ALPHA),
    ]
if TEST_REF_IMAGE:
    cmd += ["--test_ref", str(TEST_REF_IMAGE)]
if TEST_AUDIO:
    cmd += ["--test_aud", str(TEST_AUDIO)]
if TEST_EVERY_N:
    cmd += ["--test_out_dir", str(EXP_DIR / "test_videos")]
if not AUTO_RESUME:
    cmd.append("--no_auto_resume")
if RESUME_CKPT:
    cmd += ["--resume_ckpt", str(RESUME_CKPT)]
if EMOTION_FILTER:
    cmd += ["--emotion_filter"] + list(EMOTION_FILTER)

print("Launching:", " ".join(cmd[:9]), "...\n")
with open(console_log, "a") as logf:
    kwargs = dict(stdout=logf, stderr=subprocess.STDOUT, cwd=str(REPO_ROOT))
    if os.name == "posix":
        kwargs["start_new_session"] = True
    else:
        kwargs["creationflags"] = subprocess.CREATE_NEW_PROCESS_GROUP
    proc = subprocess.Popen(cmd, **kwargs)

(EXP_DIR / "train.pid").write_text(str(proc.pid))
print(f"Training started  | pid={proc.pid}")
print(f"Console log:      {console_log}")
print(f"Checkpoint dir:   {EXP_DIR / 'checkpoints'}")
print(f"Test videos:      {EXP_DIR / 'test_videos'} (every {TEST_EVERY_N} epochs)")
print("Run the MONITOR cell below to follow progress.")


In [ ]:
# OPTION B -- Run emotion LoRA training directly in this notebook (single GPU).
# This blocks the kernel until training finishes.
import importlib.util
import sys
from argparse import Namespace
from pathlib import Path

training_script = REPO_ROOT / "generator" / "run_emotion_lora_training.py"
spec = importlib.util.spec_from_file_location("run_emotion_lora_training", str(training_script))
train_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_mod)

args = Namespace(
    repo_root=str(REPO_ROOT),
    dataset_path=str(EMOTION_DATASET_DIR),
    base_generator_ckpt=str(BASE_GENERATOR_CKPT),
    style_lora_ckpt=str(STYLE_LORA_CKPT) if STYLE_LORA_CKPT and Path(STYLE_LORA_CKPT).exists() else None,
    style_lora_rank=STYLE_LORA_RANK,
    style_lora_alpha=STYLE_LORA_ALPHA,
    pretrained_ckpt=None,
    exp_root=str(EXP_ROOT),
    exp_name=EXP_NAME,
    wav2vec_model_path=str(WAV2VEC_DIR),
    renderer_ckpt=str(RENDERER_CKPT),
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    batch_size=BATCH_SIZE,
    lr=LR,
    max_steps=MAX_STEPS,
    max_epochs=MAX_EPOCHS,
    save_freq=SAVE_FREQ,
    display_freq=DISPLAY_FREQ,
    log_every_n_steps=LOG_EVERY,
    num_workers=NUM_WORKERS,
    val_split=VAL_SPLIT,
    resume_ckpt=RESUME_CKPT,
    no_auto_resume=(not AUTO_RESUME),
    precision=MIXED_PRECISION,
    list_path=None,
    val_list_path=None,
    emotion_filter=list(EMOTION_FILTER) if EMOTION_FILTER else None,
    test_ref=str(TEST_REF_IMAGE) if TEST_REF_IMAGE else None,
    test_aud=str(TEST_AUDIO) if TEST_AUDIO else None,
    test_out_dir=str(EXP_DIR / "test_videos") if TEST_EVERY_N else None,
    test_every_n_epochs=TEST_EVERY_N,
    test_crop=1 if TEST_CROP else 0,
    test_emotion_id=TEST_EMOTION_ID,
    test_cfg=TEST_CFG_SCALE,
    test_nfe=TEST_NFE,
    test_seed=TEST_SEED,
)
train_mod.main(args)


In [ ]:
# MONITOR -- re-run this cell anytime to check training progress
import os
from pathlib import Path

def _tail(path, n=20):
    path = Path(path)
    if not path.exists():
        print(f"  (no {path.name} yet)")
        return
    lines = path.read_text(errors="replace").splitlines()
    print(f"--- Last {min(n, len(lines))} lines of {path.name} ---")
    print("\n".join(lines[-n:]))

pid_file = EXP_DIR / "train.pid"
if pid_file.exists():
    pid = int(pid_file.read_text().strip())
    try:
        os.kill(pid, 0)
        print(f"Process {pid} is RUNNING [OK]")
    except OSError:
        print(f"Process {pid} is NOT running (finished or stopped)")

ckpts = sorted((EXP_DIR / "checkpoints").glob("*.ckpt"))
print(f"Checkpoints saved: {len(ckpts)}")
for c in ckpts[-3:]:
    print(f"  {c.name} ({c.stat().st_size / 1e6:.1f} MB)")

vids = sorted((EXP_DIR / "test_videos").glob("*.mp4"))
print(f"Test videos generated: {len(vids)}")
for v in vids[-3:]:
    print(f"  {v.name} ({v.stat().st_size / 1e6:.1f} MB)")

print()
_tail(EXP_DIR / "train_log.txt", 15)
print()
_tail(EXP_DIR / "train_console.log", 15)


In [ ]:
# STOP training started with OPTION A (sends SIGTERM; last checkpoint saved)
STOP_TRAINING = False

if not STOP_TRAINING:
    print("Set STOP_TRAINING = True and re-run this cell to stop background training.")
else:
    import os, signal
    pid_file = EXP_DIR / "train.pid"
    assert pid_file.exists(), "No train.pid found -- was training started with Option A?"
    pid = int(pid_file.read_text().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print(f"Sent SIGTERM to process {pid}. Last checkpoint preserved.")
    except ProcessLookupError:
        print(f"Process {pid} already stopped.")


## Validate the trained checkpoint

The cells below load the latest trained checkpoint and verify that:

1. it contains the emotion conditioner weights and the LoRA adapter keys,
2. the emotion projection actually learned (non-zero weight norm),
3. the 6 emotions produce **distinct, non-zero conditioning biases** (pairwise L2),
4. and finally render test videos for **all 6 emotions** with `3robert.jpeg` +
   `audio_3.wav`.


In [ ]:
# Load the trained emotion LoRA checkpoint and verify its contents
import sys
from pathlib import Path

import torch

for _p in (str(REPO_ROOT), str(REPO_ROOT / "generator")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

EVAL_CKPT = None  # None = auto-pick last.ckpt

if EVAL_CKPT is None:
    last_ckpt = EXP_DIR / "checkpoints" / "last.ckpt"
    step_ckpts = sorted((EXP_DIR / "checkpoints").glob("step=*.ckpt"),
                        key=lambda p: p.stat().st_mtime)
    EVAL_CKPT = last_ckpt if last_ckpt.exists() else (step_ckpts[-1] if step_ckpts else None)

assert EVAL_CKPT and Path(EVAL_CKPT).exists(), \
    f"No checkpoint found in {EXP_DIR / 'checkpoints'}"
print(f"Evaluating: {EVAL_CKPT}")

ckpt = torch.load(str(EVAL_CKPT), map_location="cpu", weights_only=False)
state = ckpt.get("ema_state_dict") or ckpt.get("state_dict", ckpt)
if isinstance(state, dict) and "model" in state and isinstance(state["model"], dict):
    state = state["model"]
state = {k.replace("model.", "", 1) if k.startswith("model.") else k: v for k, v in state.items()}

# Check emotion-specific keys
emotion_keys = [k for k in state if "emotion" in k]
lora_keys    = [k for k in state if "lora_"  in k]
print(f"\nEmotion conditioner keys: {len(emotion_keys)}")
for k in emotion_keys:
    print(f"  {k}: {tuple(state[k].shape)}")
print(f"\nLoRA adapter keys: {len(lora_keys)}")
print(f"Total keys in checkpoint: {len(state)}")

proj_w = state.get("emotion_conditioner.emotion_projection.0.weight")
if proj_w is not None:
    norm = proj_w.norm().item()
    print(f"\nemotion_projection weight norm: {norm:.6f}")
    if norm < 1e-6:
        print("  WARNING: weights still zero -- training may not have converged yet")
    else:
        print("  [OK] Emotion projection learned (norm > 0)")


In [ ]:
# Verify the emotion conditioner produces distinct, non-zero biases per emotion
import argparse
import sys

import torch

for _p in (str(REPO_ROOT), str(REPO_ROOT / "generator")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from generator.options.base_options import BaseOptions
from generator.FM import FMGenerator, EmotionConditioner, EMOTION_MAP, EMOTION_NAMES
from generator.train_lora import apply_emotion_lora_to_model

# Build model with same opt as training (no audio encoder needed on CPU)
parser = argparse.ArgumentParser()
parser = BaseOptions().initialize(parser)
parser.add_argument("--skip_fm_audio_encoder", action="store_true")
opt = parser.parse_args([])
opt.rank = "cpu"
opt.wav2vec_model_path = str(WAV2VEC_DIR)
opt.skip_fm_audio_encoder = True

model = FMGenerator(opt)
apply_emotion_lora_to_model(model, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=0.0)
missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()
print(f"Model loaded | missing={len(missing)} unexpected={len(unexpected)}")

# Check canonical 6 emotion embeddings
print("\nEmotion bias norms (T=50 frames):")
biases = {}
for idx in range(6):
    name = EMOTION_NAMES[idx]
    eid = torch.tensor([idx], dtype=torch.long)
    with torch.no_grad():
        bias = model.emotion_conditioner(eid, T=50, train=False)  # [1, 50, 32]
    norm = bias.mean(dim=1).norm().item()
    biases[name] = bias.squeeze(0).mean(0)  # [32]
    print(f"  [{idx}] {name:10s}: norm={norm:.6f}")

# Pairwise distances
print("\nPairwise L2 distances between emotion embeddings:")
names = [EMOTION_NAMES[i] for i in range(6)]
for i, n1 in enumerate(names):
    for j, n2 in enumerate(names):
        if j <= i:
            continue
        dist = (biases[n1] - biases[n2]).norm().item()
        print(f"  {n1:10s} vs {n2:10s}: {dist:.4f}")


In [ ]:
# ============================================================================
# Render test videos for ALL 6 emotions from the trained checkpoint
# Outputs to: {EXP_DIR}/final_emotion_tests/
#   0_neutral.mp4, 1_happy.mp4, 2_sad.mp4, 3_angry.mp4, 4_surprise.mp4, 5_fear.mp4
# ============================================================================
import subprocess
import sys
from pathlib import Path

test_script = REPO_ROOT / "generator" / "test_emotions.py"
test_out_dir = EXP_DIR / "final_emotion_tests"

cmd = [
    sys.executable, str(test_script),
    "--checkpoint",           str(EVAL_CKPT),
    "--base_generator_ckpt",  str(BASE_GENERATOR_CKPT),
    "--ref_img",              str(TEST_REF_IMAGE or (REPO_ROOT / "assets" / "3robert.jpeg")),
    "--audio",                str(TEST_AUDIO or (REPO_ROOT / "assets" / "audio_3.wav")),
    "--out_dir",              str(test_out_dir),
    "--cfg",                  str(TEST_CFG_SCALE),
    "--nfe",                  str(TEST_NFE),
    "--seed",                 str(TEST_SEED),
]
if STYLE_LORA_CKPT and Path(STYLE_LORA_CKPT).exists():
    cmd += ["--style_lora_ckpt", str(STYLE_LORA_CKPT)]

print("Rendering test videos for all 6 emotions ...")
subprocess.run(cmd, check=True)
print(f"\nAll 6 emotion test videos saved in: {test_out_dir}")


## After Training -- Output Files

All outputs are in `{EXP_ROOT}/{EXP_NAME}/` (e.g. `/workspace/speech2face_emotion/emotion_exps/emotion_lora_v1/`):

| Path | Content |
|------|---------|
| `checkpoints/last.ckpt` | Latest checkpoint (auto-resume source) |
| `checkpoints/step=XXXXXX.ckpt` | Periodic checkpoints (every `SAVE_FREQ` steps) |
| `train_log.txt` | Human-readable epoch/step/loss/lr log |
| `train_console.log` | Full console output (Option A) |
| `version_*/` | TensorBoard + CSV metrics |
| `test_videos/epoch_XXXX.mp4` | Test videos rendered every 10 epochs |
| `final_emotion_tests/*.mp4` | Final 6-emotion validation renders |

## Using the trained emotion adapter in the new speech2face pipeline

The checkpoint stores the **full generator weights** (base + emotion conditioner +
emotion LoRA, in both `state_dict` and `ema_state_dict` form). To deploy:

1. **Apply the same 3 patches** (`FM.py`, `FMT.py`, `train_lora.py` cells of this
   notebook) to the generator code your live server loads. The patched code is
   backward-compatible: without an `emotion_id` it behaves exactly like stock.
2. **Load the weights on top of the base generator** (same pattern as
   `generator/test_emotions.py`):

```python
model = FMGenerator(opt)                                   # base architecture
model.load_state_dict(base_generator_state, strict=False)  # generator.ckpt
apply_emotion_lora_to_model(model, rank=32, alpha=64.0, dropout=0.0)
model.load_state_dict(emotion_ckpt_state, strict=False)    # this checkpoint
```

3. **Set the emotion at inference time** by adding one entry to the data dict passed
   to `model.sample(...)` (works for both the full and chunked/streaming paths):

```python
data["emotion_id"] = 1   # 0=neutral 1=happy 2=sad 3=angry 4=surprise 5=fear
```

4. The audio path is unchanged: PersonaPlex Helium hidden states -> UniTalk adapter
   -> wav2vec2-space features -> `data["a_feat"]`. The emotion conditioning composes
   with classifier-free guidance (`a_cfg_scale`); the unconditional branch uses the
   learned null-emotion token automatically.

Because the generator is the same `FMGenerator` the new pipeline already loads (via
`--generator_path` / UniTalk adapter), no other live-system changes are required.
